In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/spikie/capstone-split/latest.pt
/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl
/kaggle/input/datasets/spikie/capstone-split/legal_bpe_64k.pkl
/kaggle/input/datasets/spikie/capstone-split/legal_m1.py
/kaggle/input/datasets/spikie/capstone-split/metadata.json
/kaggle/input/datasets/spikie/capstone-split/train_manifest.json
/kaggle/input/datasets/spikie/capstone-split/train_tokens.bin
/kaggle/input/datasets/spikie/capstone-split/valid_manifest.json
/kaggle/input/datasets/spikie/capstone-split/valid_tokens.bin


In [4]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [5]:
#Custom BPE Tokenizer adapted from A-1
from collections import Counter,defaultdict
import heapq
import re
from tqdm.auto import tqdm

class ByteLevelBPE:
    SPECIALS=["<eos>"]

    def __init__(self,vocab_size=32000):
        self.target_vocab_size=vocab_size
        self._merge_id_pairs=[]
        self.merges=[]
        self._token_bytes=[]
        self._bytes_to_id={}
        self._merge_ranks={}
        self._merge_to_new_id={}
        self.itos=[]
        self.stoi={}
        self.vocab_size=0
        self.eos_id=0

    @staticmethod
    def _get_word_pairs(symbols):
        return Counter(zip(symbols,symbols[1:]))

    @staticmethod
    def _merge_word(symbols,pair,new_id):
        left,right=pair
        merged=[]
        i=0

        while i<len(symbols):
            if i+1<len(symbols) and symbols[i]==left and symbols[i+1]==right:
                merged.append(new_id)
                i+=2
            else:
                merged.append(symbols[i])
                i+=1

        return merged

    def train_from_word_counts(self,word_freqs):
        self._token_bytes=[(byte,) for byte in range(256)]
        words=[]
        freqs=[]
        for word,freq in tqdm(word_freqs.items(),desc="Preparing BPE corpus"):
            symbols=list(word.encode("utf-8"))
            if len(symbols)>=2:
                words.append(symbols)
                freqs.append(freq)

        pair_counts=Counter()
        pair_to_words=defaultdict(set)
        for word_idx,symbols in tqdm(enumerate(words),total=len(words),desc="Building pair statistics"):
            word_freq=freqs[word_idx]
            for pair,occurrences in self._get_word_pairs(symbols).items():
                pair_counts[pair]+=occurrences*word_freq
                pair_to_words[pair].add(word_idx)

        heap=[]
        for pair,count in pair_counts.items():
            left,right=pair
            heap.append((-count,self._token_bytes[left],self._token_bytes[right],pair))
        heapq.heapify(heap)
        max_merges=self.target_vocab_size-256-len(self.SPECIALS)
        for _ in tqdm(range(max_merges),desc="Learning BPE merges"):
            best_pair=None
            while heap:
                neg_count,_,_,pair=heapq.heappop(heap)
                count=pair_counts.get(pair,0)
                if count==-neg_count and count>1:
                    best_pair=pair
                    break

            if best_pair is None:
                break

            left,right=best_pair
            new_id=len(self._token_bytes)
            self._token_bytes.append(self._token_bytes[left]+self._token_bytes[right])
            self._merge_id_pairs.append(best_pair)
            self.merges.append((self._token_bytes[left],self._token_bytes[right]))
            affected_words=list(pair_to_words.get(best_pair,set()))
            changed_pairs=set()
            for word_idx in affected_words:
                symbols=words[word_idx]
                word_freq=freqs[word_idx]
                new_symbols=[]
                i=0
                while i<len(symbols):
                    if i+1<len(symbols) and symbols[i]==left and symbols[i+1]==right:
                        new_symbols.append(new_id)
                        i+=2
                    else:
                        new_symbols.append(symbols[i])
                        i+=1

                if new_symbols==symbols:
                    continue
                old_pairs=self._get_word_pairs(symbols)
                new_pairs=self._get_word_pairs(new_symbols)
                for pair,occurrences in old_pairs.items():
                    pair_counts[pair]-=occurrences*word_freq
                    changed_pairs.add(pair)

                for pair,occurrences in new_pairs.items():
                    pair_counts[pair]+=occurrences*word_freq
                    changed_pairs.add(pair)

                for pair in old_pairs:
                    if pair not in new_pairs:
                        pair_to_words[pair].discard(word_idx)

                for pair in new_pairs:
                    pair_to_words[pair].add(word_idx)

                words[word_idx]=new_symbols
            pair_counts.pop(best_pair,None)
            pair_to_words.pop(best_pair,None)
            for pair in changed_pairs:
                count=pair_counts.get(pair,0)

                if count<=0:
                    pair_counts.pop(pair,None)
                    pair_to_words.pop(pair,None)
                else:
                    left_pair,right_pair=pair
                    heapq.heappush(heap,(-count,self._token_bytes[left_pair],self._token_bytes[right_pair],pair))

        self._bytes_to_id={token_bytes:idx for idx,token_bytes in enumerate(self._token_bytes)}
        self._merge_ranks={pair:rank for rank,pair in enumerate(self._merge_id_pairs)}
        self._merge_to_new_id={pair:256+rank for rank,pair in enumerate(self._merge_id_pairs)}
        self.itos=self.SPECIALS.copy()
        self.itos.extend(self._token_bytes)
        self.stoi={token:i for i,token in enumerate(self.itos)}
        self.eos_id=0
        self.vocab_size=len(self.itos)
        print("Learned merges:",len(self.merges))
        print("Final vocabulary size:",self.vocab_size)
        return self

    def encode_word(self,word):
        symbols=[byte for byte in word.encode("utf-8")]
        while len(symbols)>=2:
            best_rank=None
            best_pair=None
            for i in range(len(symbols)-1):
                pair=(symbols[i],symbols[i+1])
                rank=self._merge_ranks.get(pair)
                if rank is not None and (best_rank is None or rank<best_rank):
                    best_rank=rank
                    best_pair=pair

            if best_pair is None:
                break

            symbols=self._merge_word(symbols,best_pair,self._merge_to_new_id[best_pair])
        return [token_id+1 for token_id in symbols]

    def encode_text(self,text,add_eos=True):
        tokens=[]

        for part in re.findall(r"\s+|\S+",text):
            tokens.extend(self.encode_word(part))

        if add_eos:
            tokens.append(self.eos_id)

        return tokens

In [2]:
import os
import math
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist

from torch.utils.data import Dataset, DataLoader
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Dataset
DATA_DIR = "/kaggle/input/datasets/spikie/capstone-split"

TRAIN_PATH = os.path.join(DATA_DIR, "train_tokens.bin")
VALID_PATH = os.path.join(DATA_DIR, "valid_tokens.bin")

# Model configuration
VOCAB_SIZE = 32000
CONTEXT_LEN = 2048
N_LAYERS = 18
D_MODEL = 640
N_HEADS = 10
HEAD_DIM = D_MODEL // N_HEADS
D_FF = 2560

# Attention and normalization
ATTENTION_TYPE = "MQA"
POSITIONAL_ENCODING = "ALiBi"
N_KV_HEADS = 1
NORMALIZATION = "Post-LN"

# Training configuration
TARGET_TOKENS = 250_000_000
BATCH_SIZE_PER_GPU = 2
GRAD_ACCUM_STEPS = 1
LEARNING_RATE = 3e-4
MIN_LR = 3e-5
WEIGHT_DECAY = 0.1
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0

# Checkpoints
CKPT_DIR = "/kaggle/working/m2_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("\nTraining file exists:", os.path.exists(TRAIN_PATH))
print("Validation file exists:", os.path.exists(VALID_PATH))
print("Head dimension:", HEAD_DIM)
print("Checkpoint directory:", CKPT_DIR)

PyTorch: 2.11.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Training file exists: True
Validation file exists: True
Head dimension: 64
Checkpoint directory: /kaggle/working/m2_checkpoints


In [7]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


def build_alibi_slopes(n_heads):
    # Standard ALiBi slopes for an arbitrary number of attention heads.
    def slopes_power_of_two(n):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        ratio = start
        return [start * (ratio ** i) for i in range(n)]

    if math.log2(n_heads).is_integer():
        return torch.tensor(slopes_power_of_two(n_heads), dtype=torch.float32)

    closest_power = 2 ** math.floor(math.log2(n_heads))
    slopes = slopes_power_of_two(closest_power)
    extra = build_alibi_slopes(2 * closest_power)[0::2][:n_heads - closest_power].tolist()
    return torch.tensor(slopes + extra, dtype=torch.float32)



class MQA(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()

        assert d_model % n_heads == 0

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.register_buffer(
            "alibi_slopes",
            build_alibi_slopes(n_heads),
            persistent=True
        )

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(
            B, T, self.n_heads, self.head_dim
        ).transpose(1, 2)

        k = self.k_proj(x).view(
            B, T, 1, self.head_dim
        ).transpose(1, 2)

        v = self.v_proj(x).view(
            B, T, 1, self.head_dim
        ).transpose(1, 2)

        # Share the single K/V head across all query heads.
        k = k.expand(B, self.n_heads, T, self.head_dim)
        v = v.expand(B, self.n_heads, T, self.head_dim)

        # ALiBi bias: slope[h] * (key_position - query_position).
        pos = torch.arange(T, device=x.device)
        relative = pos[None, :] - pos[:, None]  # [query, key]

        slopes = self.alibi_slopes.to(
            device=x.device, dtype=torch.float32
        )

        bias = slopes[None, :, None, None] * relative[
            None, None, :, :
        ].float()

        # Future positions are masked; past positions get negative bias.
        bias = bias.masked_fill(
            relative[None, None, :, :] > 0,
            float("-inf")
        )

        attn = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=bias.to(dtype=q.dtype),
            dropout_p=self.dropout if self.training else 0.0
        )

        attn = attn.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(attn)


class M2Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.05):
        super().__init__()

        self.attn = MQA(d_model, n_heads, dropout=dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Post-LN: residual addition first, normalization afterward.
        x = self.ln1(x + self.dropout(self.attn(x)))
        x = self.ln2(x + self.dropout(self.ffn(x)))
        return x


class M2Model(nn.Module):
    def __init__(
        self,
        vocab_size=32000,
        d_model=640,
        n_layers=18,
        n_heads=10,
        d_ff=2560,
        dropout=0.05
    ):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, d_model)
        nn.init.normal_(self.token_emb.weight, mean=0.0, std=0.02)

        self.blocks = nn.ModuleList([
            M2Block(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Tie token embeddings and output projection weights.
        self.lm_head.weight = self.token_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, input_ids):
        x = self.token_emb(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.final_ln(x)
        logits = self.lm_head(x)
        return logits


# Basic shape and parameter-count test.
model = M2Model(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    d_ff=D_FF
)

parameter_count = sum(p.numel() for p in model.parameters())

print("M2 model created successfully")
print("Output shape:", model(torch.randint(
    0, VOCAB_SIZE, (2, 32)
)).shape)
print(f"Parameters: {parameter_count / 1e6:.2f}M")
print("Attention heads: 10 query, 1 key, 1 value")
print("Normalization: Post-LayerNorm")


M2 model created successfully
Output shape: torch.Size([2, 32, 32000])
Parameters: 95.79M
Attention heads: 10 query, 1 key, 1 value
Normalization: Post-LayerNorm


In [8]:

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class TokenBlockDataset(Dataset):
    def __init__(self, path, context_len):
        self.path = path
        self.context_len = context_len

        # Map the binary file without loading the entire file into RAM.
        self.tokens = np.memmap(path, dtype=np.int16, mode="r")
        self.num_tokens = len(self.tokens)
        self.num_blocks = (self.num_tokens - 1) // context_len

        print(
            f"{path.split('/')[-1]}: "
            f"{self.num_tokens:,} tokens, "
            f"{self.num_blocks:,} full blocks"
        )

    def __len__(self):
        return self.num_blocks

    def __getitem__(self, idx):
        start = idx * self.context_len
        end = start + self.context_len + 1

        chunk = np.array(self.tokens[start:end], dtype=np.int64)
        chunk = torch.from_numpy(chunk)

        x = chunk[:-1]
        y = chunk[1:]

        return x, y


train_dataset = TokenBlockDataset(TRAIN_PATH, CONTEXT_LEN)
valid_dataset = TokenBlockDataset(VALID_PATH, CONTEXT_LEN)

GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_GPU * 2  # Two GPUs
TOKENS_PER_STEP = (
    GLOBAL_BATCH_SIZE * CONTEXT_LEN * GRAD_ACCUM_STEPS
)

MAX_STEPS = math.ceil(TARGET_TOKENS / TOKENS_PER_STEP)
WARMUP_STEPS = int(MAX_STEPS * WARMUP_RATIO)

print("\nGlobal batch size:", GLOBAL_BATCH_SIZE)
print("Tokens per optimizer step:", TOKENS_PER_STEP)
print("Target training tokens:", f"{TARGET_TOKENS:,}")
print("Planned optimizer steps:", f"{MAX_STEPS:,}")
print("Warmup steps:", f"{WARMUP_STEPS:,}")

# Verify a sample before training.
x, y = train_dataset[0]

print("\nInput shape:", x.shape)
print("Target shape:", y.shape)
print("Input token range:", int(x.min()), int(x.max()))
print("Target token range:", int(y.min()), int(y.max()))


train_tokens.bin: 5,733,521,408 tokens, 2,799,570 full blocks
valid_tokens.bin: 640,489,472 tokens, 312,738 full blocks

Global batch size: 4
Tokens per optimizer step: 8192
Target training tokens: 250,000,000
Planned optimizer steps: 30,518
Warmup steps: 915

Input shape: torch.Size([2048])
Target shape: torch.Size([2048])
Input token range: 11 31137
Target token range: 11 31137


In [10]:

%%writefile /kaggle/working/m2_model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def build_alibi_slopes(n_heads):
    def slopes_power_of_two(n):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        ratio = start
        return [start * (ratio ** i) for i in range(n)]

    if math.log2(n_heads).is_integer():
        return torch.tensor(slopes_power_of_two(n_heads), dtype=torch.float32)

    closest_power = 2 ** math.floor(math.log2(n_heads))
    slopes = slopes_power_of_two(closest_power)
    extra = build_alibi_slopes(2 * closest_power)[0::2][:n_heads - closest_power].tolist()
    return torch.tensor(slopes + extra, dtype=torch.float32)


class MQA(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.register_buffer(
            "alibi_slopes", build_alibi_slopes(n_heads), persistent=True
        )

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(
            B, T, self.n_heads, self.head_dim
        ).transpose(1, 2)

        k = self.k_proj(x).view(
            B, T, 1, self.head_dim
        ).transpose(1, 2)

        v = self.v_proj(x).view(
            B, T, 1, self.head_dim
        ).transpose(1, 2)

        k = k.expand(B, self.n_heads, T, self.head_dim)
        v = v.expand(B, self.n_heads, T, self.head_dim)

        pos = torch.arange(T, device=x.device)
        relative = pos[None, :] - pos[:, None]

        slopes = self.alibi_slopes.to(
            device=x.device, dtype=torch.float32
        )
        bias = slopes[None, :, None, None] * relative[
            None, None, :, :
        ].float()

        bias = bias.masked_fill(
            relative[None, None, :, :] > 0, float("-inf")
        )

        attn = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=bias.to(dtype=q.dtype),
            dropout_p=self.dropout if self.training else 0.0
        )

        attn = attn.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(attn)


class M2Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.05):
        super().__init__()
        self.attn = MQA(d_model, n_heads, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.ln1(x + self.dropout(self.attn(x)))
        x = self.ln2(x + self.dropout(self.ffn(x)))
        return x


class M2Model(nn.Module):
    def __init__(
        self,
        vocab_size=32000,
        d_model=640,
        n_layers=18,
        n_heads=10,
        d_ff=2560,
        dropout=0.05
    ):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            M2Block(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        self.apply(self._init_weights)
        nn.init.normal_(self.token_emb.weight, mean=0.0, std=0.02)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, input_ids):
        x = self.token_emb(input_ids)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.final_ln(x))


Overwriting /kaggle/working/m2_model.py


In [11]:

import sys
sys.path.insert(0, "/kaggle/working")

from m2_model import M2Model

test_model = M2Model()
test_output = test_model(
    torch.randint(0, VOCAB_SIZE, (1, 16))
)

print("Model import successful")
print("Output shape:", test_output.shape)
print("Parameters:", f"{sum(p.numel() for p in test_model.parameters()) / 1e6:.2f}M")

del test_model, test_output
torch.cuda.empty_cache()


Model import successful
Output shape: torch.Size([1, 16, 32000])
Parameters: 95.79M


In [16]:

%%writefile /kaggle/working/train_m2.py
import os
import time
import math
import random
import numpy as np

import torch
import torch.nn.functional as F
import torch.distributed as dist

from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.amp import autocast, GradScaler
from tqdm import tqdm

from m2_model import M2Model


# ---------------- Configuration ----------------

SEED = 42
DATA_DIR = "/kaggle/input/datasets/spikie/capstone-split"
TRAIN_PATH = os.path.join(DATA_DIR, "train_tokens.bin")
VALID_PATH = os.path.join(DATA_DIR, "valid_tokens.bin")

VOCAB_SIZE = 32000
CONTEXT_LEN = 2048
D_MODEL = 640
N_LAYERS = 18
N_HEADS = 10
D_FF = 2560
DROPOUT = 0.05

TARGET_TOKENS = 250_000_000
BATCH_SIZE_PER_GPU = 2
GRAD_ACCUM_STEPS = 1
TOKENS_PER_STEP = 2 * BATCH_SIZE_PER_GPU * CONTEXT_LEN * GRAD_ACCUM_STEPS
MAX_STEPS = math.ceil(TARGET_TOKENS / TOKENS_PER_STEP)

LR = 3e-4
MIN_LR = 3e-5
WEIGHT_DECAY = 0.1
WARMUP_STEPS = int(0.03 * MAX_STEPS)
MAX_GRAD_NORM = 1.0

SAVE_EVERY = 500
VAL_EVERY = 1000
VAL_BATCHES = 50

SMOKE_TEST = False
SMOKE_STEPS = 5

CKPT_DIR = "/kaggle/working/m2_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)


# ---------------- Dataset ----------------

class TokenBlockDataset(Dataset):
    def __init__(self, path, context_len):
        self.tokens = np.memmap(path, dtype=np.int16, mode="r")
        self.context_len = context_len
        self.num_blocks = (len(self.tokens) - 1) // context_len

    def __len__(self):
        return self.num_blocks

    def __getitem__(self, idx):
        start = idx * self.context_len
        chunk = np.array(
            self.tokens[start:start + self.context_len + 1],
            dtype=np.int64
        )
        chunk = torch.from_numpy(chunk)
        return chunk[:-1], chunk[1:]


# ---------------- Training ----------------

def main():
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    dist.init_process_group(backend="nccl")

    torch.manual_seed(SEED + rank)
    np.random.seed(SEED + rank)
    random.seed(SEED + rank)
    torch.cuda.manual_seed_all(SEED + rank)

    train_dataset = TokenBlockDataset(TRAIN_PATH, CONTEXT_LEN)
    valid_dataset = TokenBlockDataset(VALID_PATH, CONTEXT_LEN)

    train_sampler = DistributedSampler(
        train_dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        seed=SEED,
        drop_last=True
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE_PER_GPU,
        sampler=train_sampler,
        num_workers=2,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True
    )

    valid_sampler = DistributedSampler(
        valid_dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=False,
        drop_last=False
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE_PER_GPU,
        sampler=valid_sampler,
        num_workers=2,
        pin_memory=True,
        drop_last=False,
        persistent_workers=True
    )

    model = M2Model(
        vocab_size=VOCAB_SIZE,
        d_model=D_MODEL,
        n_layers=N_LAYERS,
        n_heads=N_HEADS,
        d_ff=D_FF,
        dropout=DROPOUT
    ).to(device)

    model = DDP(model, device_ids=[local_rank])

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        fused=True
    )

    scaler = GradScaler("cuda", enabled=True)

    start_step = 0
    best_val = float("inf")
    latest_path = os.path.join(CKPT_DIR, "latest.pt")

    if os.path.exists(latest_path):
        checkpoint = torch.load(latest_path, map_location="cpu")
        model.module.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scaler.load_state_dict(checkpoint["scaler"])
        start_step = checkpoint["step"]
        best_val = checkpoint.get("best_val", float("inf"))

        if rank == 0:
            print(f"Resuming M2 from step {start_step:,}")

    max_steps = SMOKE_STEPS if SMOKE_TEST else MAX_STEPS

    def get_lr(step):
        if step < WARMUP_STEPS:
            return LR * (step + 1) / max(1, WARMUP_STEPS)

        progress = min(
            1.0,
            (step - WARMUP_STEPS) /
            max(1, MAX_STEPS - WARMUP_STEPS)
        )

        return MIN_LR + 0.5 * (LR - MIN_LR) * (
            1 + math.cos(math.pi * progress)
        )

    @torch.no_grad()
    def validate():
        model.eval()
        total_loss = torch.zeros((), device=device)
        total_tokens = torch.zeros((), device=device)

        for batch_idx, (x, y) in enumerate(valid_loader):
            if batch_idx >= VAL_BATCHES:
                break

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with autocast("cuda", dtype=torch.float16):
                logits = model(x)
                loss = F.cross_entropy(
                    logits.reshape(-1, VOCAB_SIZE).float(),
                    y.reshape(-1),
                    reduction="sum"
                )

            total_loss += loss.detach()
            total_tokens += y.numel()

        dist.all_reduce(total_loss, op=dist.ReduceOp.SUM)
        dist.all_reduce(total_tokens, op=dist.ReduceOp.SUM)

        model.train()
        return (total_loss / total_tokens).item()

    if rank == 0:
        params = sum(p.numel() for p in model.module.parameters())
        print(f"M2 parameters: {params / 1e6:.2f}M")
        print(f"Target tokens: {TARGET_TOKENS:,}")
        print(f"Tokens per step: {TOKENS_PER_STEP:,}")
        print(f"Maximum steps: {max_steps:,}")

    progress_bar = tqdm(
        total=max_steps,
        initial=start_step,
        desc="M2 training",
        disable=(rank != 0)
    )

    running_loss = 0.0
    log_count = 0
    last_log_time = time.time()
    last_log_step = start_step
    start_time = time.time()

    optimizer.zero_grad(set_to_none=True)

    # Create one persistent iterator and restart only at epoch boundaries.
    epoch = 0
    train_sampler.set_epoch(epoch)
    train_iter = iter(train_loader)

    for step in range(start_step, max_steps):
        try:
            x, y = next(train_iter)
        except StopIteration:
            epoch += 1
            train_sampler.set_epoch(epoch)
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        lr_now = get_lr(step)
        for group in optimizer.param_groups:
            group["lr"] = lr_now

        with autocast("cuda", dtype=torch.float16):
            logits = model(x)
            loss = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE).float(),
                y.reshape(-1)
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), MAX_GRAD_NORM
        )

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        running_loss += loss.detach().item()
        log_count += 1
        progress_bar.update(1)

        if rank == 0 and (step + 1) % 10 == 0:
            now = time.time()
            elapsed = now - last_log_time
            steps_done = (step + 1) - last_log_step
            steps_per_sec = steps_done / max(elapsed, 1e-9)
            tokens_per_sec = steps_per_sec * TOKENS_PER_STEP
            avg_loss = running_loss / max(1, log_count)
            remaining = max_steps - (step + 1)
            eta_minutes = remaining / max(steps_per_sec, 1e-9) / 60

            progress_bar.set_postfix(
                loss=f"{avg_loss:.4f}",
                lr=f"{lr_now:.2e}",
                tok_s=f"{tokens_per_sec:.0f}",
                eta_min=f"{eta_minutes:.1f}"
            )

            running_loss = 0.0
            log_count = 0
            last_log_time = now
            last_log_step = step + 1

        if (step + 1) % VAL_EVERY == 0 or step + 1 == max_steps:
            val_loss = validate()

            if rank == 0:
                print(
                    f"\nStep {step + 1:,}/{max_steps:,} | "
                    f"Validation loss: {val_loss:.4f} | "
                    f"Perplexity: {math.exp(min(val_loss, 20)):.3f}"
                )

                if val_loss < best_val:
                    best_val = val_loss
                    torch.save(
                        {
                            "step": step + 1,
                            "model": model.module.state_dict(),
                            "val_loss": val_loss,
                            "best_val": best_val
                        },
                        os.path.join(CKPT_DIR, "best.pt")
                    )
                    print("Saved new best checkpoint.")

        if (step + 1) % SAVE_EVERY == 0 or step + 1 == max_steps:
            if rank == 0:
                temp_path = os.path.join(CKPT_DIR, "latest_tmp.pt")
                torch.save(
                    {
                        "step": step + 1,
                        "model": model.module.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "scaler": scaler.state_dict(),
                        "best_val": best_val
                    },
                    temp_path
                )
                os.replace(temp_path, latest_path)

    progress_bar.close()
    dist.barrier()

    if rank == 0:
        print("M2 training run finished.")
        print(f"Best validation loss: {best_val:.4f}")
        print(f"Elapsed minutes: {(time.time() - start_time) / 60:.1f}")

    dist.destroy_process_group()


if __name__ == "__main__":
    main()


Overwriting /kaggle/working/train_m2.py


In [17]:
!torchrun --nproc_per_node=2 /kaggle/working/train_m2.py

W1016 08:31:03.867000 235 torch/distributed/run.py:851] 
W1016 08:31:03.867000 235 torch/distributed/run.py:851] *****************************************
W1016 08:31:03.867000 235 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1016 08:31:03.867000 235 torch/distributed/run.py:851] *****************************************
Resuming M2 from step 5
M2 parameters: 95.79M
Target tokens: 250,000,000
Tokens per step: 8,192
Maximum steps: 30,518
M2 training:   3%| | 1000/30518 [09:47<4:52:10,  1.68it/s, eta_min=292.3, loss=3
Step 1,000/30,518 | Validation loss: 3.0694 | Perplexity: 21.528
Saved new best checkpoint.
M2 training:   7%| | 2000/30518 [19:57<4:43:36,  1.68it/s, eta_min=283.8, loss=2
Step 2,000/30,518 | Validation loss: 2.6686 | Perplexity: 14.420
Saved new best checkpoint.
M2 training:  10%

Phase 3

In [25]:
import os
from huggingface_hub import snapshot_download

DATA_DIR = "/kaggle/working/phase3_data"

snapshot_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    repo_type="dataset",
    allow_patterns=["Phase_3_RR/**"],
    local_dir=DATA_DIR
)

print("Dataset downloaded.")
for root, dirs, files in os.walk(os.path.join(DATA_DIR, "Phase_3_RR")):
    for file in files:
        print(os.path.join(root, file))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Dataset downloaded.
/kaggle/working/phase3_data/Phase_3_RR/Test/IT_test.json
/kaggle/working/phase3_data/Phase_3_RR/Test/CL_test.json
/kaggle/working/phase3_data/Phase_3_RR/Train/label_vocab.json
/kaggle/working/phase3_data/Phase_3_RR/Train/IT_train.json
/kaggle/working/phase3_data/Phase_3_RR/Train/CL_train.json
/kaggle/working/phase3_data/Phase_3_RR/Train/IT_dev.json
/kaggle/working/phase3_data/Phase_3_RR/Train/CL_dev.json


In [26]:
import os, json

DATA_DIR = "/kaggle/working/phase3_data/Phase_3_RR"
TRAIN_DIR = os.path.join(DATA_DIR, "Train")
TEST_DIR = os.path.join(DATA_DIR, "Test")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

CL_train = load_json(os.path.join(TRAIN_DIR, "CL_train.json"))
CL_dev = load_json(os.path.join(TRAIN_DIR, "CL_dev.json"))
IT_train = load_json(os.path.join(TRAIN_DIR, "IT_train.json"))
IT_dev = load_json(os.path.join(TRAIN_DIR, "IT_dev.json"))
CL_test = load_json(os.path.join(TEST_DIR, "CL_test.json"))
IT_test = load_json(os.path.join(TEST_DIR, "IT_test.json"))
label_vocab = load_json(os.path.join(TRAIN_DIR, "label_vocab.json"))

print("Label vocabulary:", label_vocab)
print("CL train/dev/test:", len(CL_train), len(CL_dev), len(CL_test))
print("IT train/dev/test:", len(IT_train), len(IT_dev), len(IT_test))
print("Sample document keys:", CL_train[0].keys())
print("Sample sentence:", CL_train[0]["text"][0])
print("Sample label:", CL_train[0]["labels"][0])

Label vocabulary: {'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}
CL train/dev/test: 40 5 5
IT train/dev/test: 40 5 5
Sample document keys: dict_keys(['id', 'text', 'labels'])
Sample sentence: Consequent upon the repeal of the MRTP Act this complaint has been received by transfer on 04.03.2010 from the office of DGIR, MRTP Commission under Section 66(6) of the Competition Act, 2002.
Sample label: Fact


In [29]:
print("Tokenizer class:", type(tokenizer).__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("EOS ID:", tokenizer.eos_id)

sample_text = CL_train[0]["text"][0]
sample_ids = tokenizer.encode_text(sample_text, add_eos=True)

print("Sample token count:", len(sample_ids))
print("Ends with EOS:", sample_ids[-1] == tokenizer.eos_id)

NameError: name 'tokenizer' is not defined

In [30]:
tokenizer = ByteLevelBPE(vocab_size=32000)

TOKENIZER_PATH = "/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl"
import pickle, os

assert os.path.exists(TOKENIZER_PATH), f"Tokenizer file not found: {TOKENIZER_PATH}"
with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

print("Tokenizer:", type(tokenizer).__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("EOS ID:", tokenizer.eos_id)

sample_text = CL_train[0]["text"][0]
sample_ids = tokenizer.encode_text(sample_text, add_eos=True)
print("Sample token count:", len(sample_ids))
print("Ends with EOS:", sample_ids[-1] == tokenizer.eos_id)

Tokenizer: ByteLevelBPE
Vocabulary size: 32000
EOS ID: 0
Sample token count: 70
Ends with EOS: True


In [31]:
from collections import Counter
import numpy as np

label_names = {v: k for k, v in label_vocab.items()}

for name, docs in [("CL train", CL_train), ("CL dev", CL_dev),
                   ("IT train", IT_train), ("IT dev", IT_dev)]:
    label_counts = Counter()
    lengths = []
    total_sentences = 0

    for doc in docs:
        for sentence, label in zip(doc["text"], doc["labels"]):
            total_sentences += 1
            label_counts[label] += 1
            lengths.append(len(tokenizer.encode_text(sentence, add_eos=True)))

    print(f"\n{name}: {len(docs)} documents, {total_sentences} sentences")
    print("Label counts:", {label_names.get(k, k): v for k, v in sorted(label_counts.items())})
    print("Token lengths — median:", int(np.median(lengths)),
          "| p90:", int(np.percentile(lengths, 90)),
          "| p95:", int(np.percentile(lengths, 95)),
          "| max:", max(lengths))


CL train: 40 documents, 10593 sentences
Label counts: {'ArgumentPetitioner': 976, 'ArgumentRespondent': 996, 'Dissent': 284, 'Fact': 2979, 'Issue': 17, 'None': 53, 'PrecedentNotReliedUpon': 209, 'PrecedentReliedUpon': 1305, 'RatioOfTheDecision': 2856, 'RulingByLowerCourt': 438, 'RulingByPresentCourt': 113, 'Statute': 367}
Token lengths — median: 58 | p90: 123 | p95: 151 | max: 1542

CL dev: 5 documents, 1396 sentences
Label counts: {'ArgumentPetitioner': 344, 'ArgumentRespondent': 236, 'Fact': 249, 'Issue': 3, 'None': 2, 'PrecedentReliedUpon': 142, 'RatioOfTheDecision': 288, 'RulingByLowerCourt': 56, 'RulingByPresentCourt': 18, 'Statute': 58}
Token lengths — median: 61 | p90: 122 | p95: 155 | max: 744

IT train: 40 documents, 6252 sentences
Label counts: {'ArgumentPetitioner': 637, 'ArgumentRespondent': 352, 'Fact': 1119, 'Issue': 90, 'None': 244, 'PrecedentNotReliedUpon': 360, 'PrecedentOverruled': 3, 'PrecedentReliedUpon': 640, 'RatioOfTheDecision': 1445, 'RulingByLowerCourt': 436, 

In [35]:
import os, torch

CHECKPOINT_PATH = "/kaggle/working/m2_checkpoints/latest.pt"
MODEL_CODE_PATH = "/kaggle/working/m2_model.py"

print("Checkpoint exists:", os.path.exists(CHECKPOINT_PATH))
print("Model code exists:", os.path.exists(MODEL_CODE_PATH))

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    print("Checkpoint keys:", list(ckpt.keys()))
    print("Training step:", ckpt.get("step"))
    print("Model tensors:", len(ckpt["model"]))
    print("First parameter keys:")
    for key in list(ckpt["model"].keys())[:12]:
        print(key, tuple(ckpt["model"][key].shape))

Checkpoint exists: True
Model code exists: True
Checkpoint keys: ['step', 'model', 'optimizer', 'scaler', 'best_val']
Training step: 30518
Model tensors: 238
First parameter keys:
token_emb.weight (32000, 640)
blocks.0.attn.alibi_slopes (10,)
blocks.0.attn.q_proj.weight (640, 640)
blocks.0.attn.k_proj.weight (64, 640)
blocks.0.attn.v_proj.weight (64, 640)
blocks.0.attn.out_proj.weight (640, 640)
blocks.0.ln1.weight (640,)
blocks.0.ln1.bias (640,)
blocks.0.ln2.weight (640,)
blocks.0.ln2.bias (640,)
blocks.0.ffn.0.weight (2560, 640)
blocks.0.ffn.0.bias (2560,)


In [34]:
import sys, torch, torch.nn as nn

MODEL_CODE_PATH = "/kaggle/working/m2_model.py"
CHECKPOINT_PATH = "/kaggle/working/m2_checkpoints/latest.pt"

with open(MODEL_CODE_PATH, "r", encoding="utf-8") as f:
    exec(compile(f.read(), MODEL_CODE_PATH, "exec"), globals())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)

backbone = M2Model(
    vocab_size=32000,
    d_model=640,
    n_layers=18,
    n_heads=10,
    d_ff=2560,
    dropout=0.05
)
result = backbone.load_state_dict(ckpt["model"], strict=True)
backbone.config = {
    "vocab_size": 32000, "d_model": 640, "n_layers": 18,
    "n_heads": 10, "d_ff": 2560, "dropout": 0.05
}

print("Checkpoint loaded:", result)
print("Parameters:", sum(p.numel() for p in backbone.parameters()))
print("Device available:", device)
print("Attention projections:", [
    name for name, _ in backbone.named_modules()
    if name.endswith(("q_proj", "k_proj", "v_proj", "out_proj"))
][:8])

Checkpoint loaded: <All keys matched successfully>
Parameters: 95787520
Device available: cuda
Attention projections: ['blocks.0.attn.q_proj', 'blocks.0.attn.k_proj', 'blocks.0.attn.v_proj', 'blocks.0.attn.out_proj', 'blocks.1.attn.q_proj', 'blocks.1.attn.k_proj', 'blocks.1.attn.v_proj', 'blocks.1.attn.out_proj']


In [36]:
import math

NUM_LABELS = len(label_vocab)
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

class LoRALinear(nn.Module):
    def __init__(self, base_layer, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base_layer
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False
        self.r = r
        self.scaling = alpha / r
        self.lora_dropout = nn.Dropout(dropout)
        self.lora_A = nn.Parameter(torch.empty(r, base_layer.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base_layer.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        return self.base(x) + (self.lora_dropout(x) @ self.lora_A.T @ self.lora_B.T) * self.scaling

class M2RhetoricalClassifier(nn.Module):
    def __init__(self, backbone, d_model, num_labels, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, num_labels))

    def forward(self, input_ids, attention_mask):
        x = self.backbone.token_emb(input_ids)
        for block in self.backbone.blocks:
            x = block(x)
        x = self.backbone.final_ln(x)
        last_idx = attention_mask.sum(dim=1) - 1
        pooled = x[torch.arange(x.size(0), device=x.device), last_idx]
        return self.classifier(pooled)

for p in backbone.parameters():
    p.requires_grad = False

for block in backbone.blocks:
    for name in ["q_proj", "k_proj", "v_proj", "out_proj"]:
        layer = getattr(block.attn, name)
        setattr(block.attn, name, LoRALinear(layer, LORA_R, LORA_ALPHA, LORA_DROPOUT))

model = M2RhetoricalClassifier(backbone, 640, NUM_LABELS).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Trainable percentage:", f"{100 * trainable_params / total_params:.2f}%")
print("LoRA modules:", sum(isinstance(m, LoRALinear) for m in model.modules()))
print("Classifier output labels:", NUM_LABELS)

Total parameters: 96,367,245
Trainable parameters: 579,725
Trainable percentage: 0.60%
LoRA modules: 72
Classifier output labels: 13


In [38]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import random

MAX_LEN = 256
PAD_ID = tokenizer.eos_id

class RhetoricalDataset(Dataset):
    def __init__(self, docs, tokenizer, label_vocab, max_len=256):
        self.examples = []
        self.tokenizer = tokenizer
        self.label_vocab = label_vocab
        self.max_len = max_len

        for doc in docs:
            for sentence, label in zip(doc["text"], doc["labels"]):
                ids = tokenizer.encode_text(sentence, add_eos=True)
                if len(ids) > max_len:
                    ids = ids[:max_len-1] + [tokenizer.eos_id]
                self.examples.append((ids, label_vocab[label]))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ids, label = self.examples[idx]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

def collate_rr(batch):
    ids, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in ids], dtype=torch.long)
    input_ids = pad_sequence(ids, batch_first=True, padding_value=PAD_ID)
    positions = torch.arange(input_ids.size(1)).unsqueeze(0)
    attention_mask = (positions < lengths.unsqueeze(1)).long()
    return input_ids, attention_mask, torch.stack(labels)

CL_train_ds = RhetoricalDataset(CL_train, tokenizer, label_vocab, MAX_LEN)
CL_dev_ds = RhetoricalDataset(CL_dev, tokenizer, label_vocab, MAX_LEN)
IT_train_ds = RhetoricalDataset(IT_train, tokenizer, label_vocab, MAX_LEN)
IT_dev_ds = RhetoricalDataset(IT_dev, tokenizer, label_vocab, MAX_LEN)

print("Dataset sizes:", len(CL_train_ds), len(CL_dev_ds), len(IT_train_ds), len(IT_dev_ds))
print("Example token length:", len(CL_train_ds[0][0]))
print("Example label:", CL_train_ds[0][1].item(), label_names[CL_train_ds[0][1].item()])

Dataset sizes: 10593 1396 6252 746
Example token length: 70
Example label: 0 Fact


In [39]:
from collections import Counter
import torch

BATCH_SIZE = 16
NUM_LABELS = len(label_vocab)

CL_train_loader = DataLoader(CL_train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_rr)
IT_train_loader = DataLoader(IT_train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_rr)

batch_ids, batch_mask, batch_labels = next(iter(CL_train_loader))
print("Input shape:", tuple(batch_ids.shape))
print("Mask shape:", tuple(batch_mask.shape))
print("Labels shape:", tuple(batch_labels.shape))
print("Mask values:", torch.unique(batch_mask).tolist())
print("Batch label IDs:", torch.unique(batch_labels).tolist())

counts = Counter()
for ds in [CL_train_ds, IT_train_ds]:
    for _, label in ds:
        counts[label.item()] += 1

class_counts = torch.tensor([counts[i] for i in range(NUM_LABELS)], dtype=torch.float32)
class_weights = class_counts.sum() / (NUM_LABELS * class_counts.clamp(min=1))
class_weights = class_weights / class_weights.mean()

print("Training label counts:", {label_names[i]: int(class_counts[i]) for i in range(NUM_LABELS)})
print("Class weights:", [round(x, 3) for x in class_weights.tolist()])
print("Total training examples:", int(class_counts.sum()))

Input shape: (16, 183)
Mask shape: (16, 183)
Labels shape: (16,)
Mask values: [0, 1]
Batch label IDs: [0, 3, 5, 9, 10]
Training label counts: {'Fact': 4098, 'Issue': 107, 'ArgumentPetitioner': 1613, 'ArgumentRespondent': 1348, 'Statute': 1099, 'Dissent': 284, 'PrecedentReliedUpon': 1945, 'PrecedentNotReliedUpon': 569, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 874, 'RatioOfTheDecision': 4301, 'RulingByPresentCourt': 307, 'None': 297}
Class weights: [0.009, 0.338, 0.022, 0.027, 0.033, 0.128, 0.019, 0.064, 12.071, 0.041, 0.008, 0.118, 0.122]
Total training examples: 16845


In [40]:
import os, json, pickle, torch

OUT_DIR = "/kaggle/working/phase3_rr/experiment_2_m2_lora"
os.makedirs(OUT_DIR, exist_ok=True)

with open(os.path.join(OUT_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

with open(os.path.join(OUT_DIR, "label_vocab.json"), "w", encoding="utf-8") as f:
    json.dump(label_vocab, f, ensure_ascii=False, indent=2)

config = {
    "model_name": "M2",
    "method": "LoRA",
    "checkpoint_path": CHECKPOINT_PATH,
    "model_code_path": MODEL_CODE_PATH,
    "vocab_size": 32000,
    "d_model": 640,
    "n_layers": 18,
    "n_heads": 10,
    "d_ff": 2560,
    "dropout": 0.05,
    "num_labels": NUM_LABELS,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "pad_id": PAD_ID,
    "class_weights": class_weights.tolist()
}
with open(os.path.join(OUT_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

datasets = {
    "CL_train": CL_train_ds,
    "CL_dev": CL_dev_ds,
    "IT_train": IT_train_ds,
    "IT_dev": IT_dev_ds
}
for name, ds in datasets.items():
    torch.save(
        [(ids, label) for ids, label in ds],
        os.path.join(OUT_DIR, f"{name}.pt")
    )

print("Saved to:", OUT_DIR)
print("Files:", sorted(os.listdir(OUT_DIR)))
print("Dataset sizes:", {name: len(ds) for name, ds in datasets.items()})

Saved to: /kaggle/working/phase3_rr/experiment_2_m2_lora
Files: ['CL_dev.pt', 'CL_train.pt', 'IT_dev.pt', 'IT_train.pt', 'config.json', 'label_vocab.json', 'tokenizer.pkl']
Dataset sizes: {'CL_train': 10593, 'CL_dev': 1396, 'IT_train': 6252, 'IT_dev': 746}


In [41]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

print("Distributed backend available:", torch.distributed.is_available())
print("NCCL available:", torch.distributed.is_nccl_available())

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Distributed backend available: True
NCCL available: True


In [42]:
%%writefile /kaggle/working/train_rr_m2_lora.py
import os, json, math, argparse, random, pickle, importlib.util
import numpy as np
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.utils.data.distributed import DistributedSampler
from torch.nn.utils.rnn import pad_sequence
from torch.nn.parallel import DistributedDataParallel as DDP
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--smoke", action="store_true")
    p.add_argument("--epochs", type=int, default=5)
    p.add_argument("--batch-size", type=int, default=16)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--patience", type=int, default=2)
    return p.parse_args()

args = parse_args()
LOCAL_RANK = int(os.environ["LOCAL_RANK"])
RANK = int(os.environ["RANK"])
WORLD_SIZE = int(os.environ["WORLD_SIZE"])
torch.cuda.set_device(LOCAL_RANK)
dist.init_process_group(backend="nccl")
device = torch.device("cuda", LOCAL_RANK)
is_main = RANK == 0

OUT_DIR = "/kaggle/working/phase3_rr/experiment_2_m2_lora"
with open(os.path.join(OUT_DIR, "config.json")) as f:
    cfg = json.load(f)
with open(os.path.join(OUT_DIR, "label_vocab.json")) as f:
    label_vocab = json.load(f)
label_vocab = {k: int(v) for k, v in label_vocab.items()}
NUM_LABELS = len(label_vocab)
PAD_ID = cfg["pad_id"]
MAX_LEN = cfg["max_len"]
os.makedirs(OUT_DIR, exist_ok=True)

def seed_everything(seed):
    random.seed(seed + RANK)
    np.random.seed(seed + RANK)
    torch.manual_seed(seed + RANK)
    torch.cuda.manual_seed_all(seed + RANK)

seed_everything(42)

with open(cfg["model_code_path"], "r", encoding="utf-8") as f:
    exec(compile(f.read(), cfg["model_code_path"], "exec"), globals())

ckpt = torch.load(cfg["checkpoint_path"], map_location="cpu", weights_only=False)
backbone = M2Model(
    vocab_size=cfg["vocab_size"], d_model=cfg["d_model"],
    n_layers=cfg["n_layers"], n_heads=cfg["n_heads"],
    d_ff=cfg["d_ff"], dropout=cfg["dropout"]
)
backbone.load_state_dict(ckpt["model"], strict=True)
del ckpt

class LoRALinear(nn.Module):
    def __init__(self, base_layer, r, alpha, dropout):
        super().__init__()
        self.base = base_layer
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False
        self.scaling = alpha / r
        self.lora_dropout = nn.Dropout(dropout)
        self.lora_A = nn.Parameter(torch.empty(r, base_layer.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base_layer.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        return self.base(x) + (self.lora_dropout(x) @ self.lora_A.T @ self.lora_B.T) * self.scaling

for p in backbone.parameters():
    p.requires_grad = False
for block in backbone.blocks:
    for name in ["q_proj", "k_proj", "v_proj", "out_proj"]:
        setattr(block.attn, name, LoRALinear(
            getattr(block.attn, name), cfg["lora_r"],
            cfg["lora_alpha"], cfg["lora_dropout"]
        ))

class Classifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(nn.Dropout(0.1), nn.Linear(cfg["d_model"], NUM_LABELS))

    def forward(self, input_ids, attention_mask):
        x = self.backbone.token_emb(input_ids)
        for block in self.backbone.blocks:
            x = block(x)
        x = self.backbone.final_ln(x)
        last_idx = attention_mask.sum(1) - 1
        pooled = x[torch.arange(x.size(0), device=x.device), last_idx]
        return self.classifier(pooled)

model = Classifier(backbone).to(device)
model = DDP(model, device_ids=[LOCAL_RANK], output_device=LOCAL_RANK, find_unused_parameters=False)
trainable = [p for p in model.parameters() if p.requires_grad]
if is_main:
    total = sum(p.numel() for p in model.parameters())
    ntrain = sum(p.numel() for p in trainable)
    print(f"Parameters: {total:,}; trainable: {ntrain:,} ({100*ntrain/total:.2f}%)")

class EncodedDataset(Dataset):
    def __init__(self, filename):
        self.data = torch.load(os.path.join(OUT_DIR, filename), map_location="cpu", weights_only=False)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        ids, label = self.data[i]
        return torch.as_tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

def collate(batch):
    ids, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in ids])
    input_ids = pad_sequence(ids, batch_first=True, padding_value=PAD_ID)
    positions = torch.arange(input_ids.size(1)).unsqueeze(0)
    mask = (positions < lengths.unsqueeze(1)).long()
    return input_ids, mask, torch.stack(labels)

CL_train = EncodedDataset("CL_train.pt")
IT_train = EncodedDataset("IT_train.pt")
CL_dev = EncodedDataset("CL_dev.pt")
IT_dev = EncodedDataset("IT_dev.pt")
train_ds = ConcatDataset([CL_train, IT_train])
train_sampler = DistributedSampler(train_ds, num_replicas=WORLD_SIZE, rank=RANK, shuffle=True, drop_last=True)
train_loader = DataLoader(
    train_ds, batch_size=args.batch_size, sampler=train_sampler,
    num_workers=2, pin_memory=True, collate_fn=collate
)
def make_eval_loader(ds):
    sampler = DistributedSampler(ds, num_replicas=WORLD_SIZE, rank=RANK, shuffle=False, drop_last=False)
    return DataLoader(ds, batch_size=args.batch_size, sampler=sampler, num_workers=2, pin_memory=True, collate_fn=collate)

CL_dev_loader = make_eval_loader(CL_dev)
IT_dev_loader = make_eval_loader(IT_dev)

weights = torch.tensor(cfg["class_weights"], dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(trainable, lr=args.lr, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda")
use_amp = True

def evaluate(loader):
    model.eval()
    loss_sum = 0.0
    local_n = 0
    y_true, y_pred = [], []
    with torch.no_grad():
        for ids, mask, labels in loader:
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(ids, mask)
                loss = criterion(logits, labels)
            loss_sum += loss.item() * labels.size(0)
            local_n += labels.size(0)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(logits.argmax(-1).cpu().tolist())
    # Gather variable-length predictions across ranks and remove sampler padding duplicates.
    gathered = [None for _ in range(WORLD_SIZE)]
    dist.all_gather_object(gathered, (y_true, y_pred, loss_sum, local_n))
    if is_main:
        true_all, pred_all = [], []
        for yt, yp, _, _ in gathered:
            true_all.extend(yt)
            pred_all.extend(yp)
        true_all = true_all[:len(loader.dataset)]
        pred_all = pred_all[:len(loader.dataset)]
        macro = f1_score(true_all, pred_all, labels=list(range(NUM_LABELS)), average="macro", zero_division=0)
        acc = accuracy_score(true_all, pred_all)
        loss_total = sum(x[2] for x in gathered) / max(1, sum(x[3] for x in gathered))
        return macro, acc, loss_total
    return None

def save_trainable(path):
    state = model.module.state_dict()
    # Save full state for a self-contained inference checkpoint, including the base weights.
    torch.save({"model": state, "config": cfg, "label_vocab": label_vocab}, path)

best_score = -1.0
best_epoch = 0
bad_epochs = 0
history = []
epochs = 1 if args.smoke else args.epochs
for epoch in range(epochs):
    model.train()
    train_sampler.set_epoch(epoch)
    running_loss = 0.0
    seen = 0
    iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", disable=not is_main)
    for ids, mask, labels in iterator:
        ids, mask, labels = ids.to(device, non_blocking=True), mask.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(ids, mask)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * labels.size(0)
        seen += labels.size(0)
        if is_main:
            iterator.set_postfix(loss=f"{running_loss/max(1,seen):.4f}")
    # DDP-safe global mean training loss
    stats = torch.tensor([running_loss, seen], dtype=torch.float64, device=device)
    dist.all_reduce(stats)
    train_loss = (stats[0] / stats[1]).item()

    cl_result = evaluate(CL_dev_loader)
    it_result = evaluate(IT_dev_loader)
    if is_main:
        cl_f1, cl_acc, cl_loss = cl_result
        it_f1, it_acc, it_loss = it_result
        score = (cl_f1 + it_f1) / 2
        record = {
            "epoch": epoch+1, "train_loss": train_loss,
            "CL_macro_f1": cl_f1, "CL_accuracy": cl_acc, "CL_loss": cl_loss,
            "IT_macro_f1": it_f1, "IT_accuracy": it_acc, "IT_loss": it_loss,
            "mean_macro_f1": score
        }
        history.append(record)
        print(json.dumps(record))
        with open(os.path.join(OUT_DIR, "history.json"), "w") as f:
            json.dump(history, f, indent=2)
        if score > best_score:
            best_score = score
            best_epoch = epoch+1
            bad_epochs = 0
            save_trainable(os.path.join(OUT_DIR, "best.pt"))
            print(f"Saved best checkpoint at epoch {best_epoch}, mean macro-F1={best_score:.4f}")
        else:
            bad_epochs += 1
    stop = torch.tensor([int(is_main and bad_epochs >= args.patience)], device=device)
    dist.broadcast(stop, src=0)
    if stop.item():
        break

if is_main:
    print(f"Training finished. Best epoch={best_epoch}, mean macro-F1={best_score:.4f}")
dist.destroy_process_group()

Writing /kaggle/working/train_rr_m2_lora.py


In [44]:
path = "/kaggle/working/train_rr_m2_lora.py"

with open(path, "r", encoding="utf-8") as f:
    script = f.read()

old_loader = '''def make_eval_loader(ds):
    sampler = DistributedSampler(ds, num_replicas=WORLD_SIZE, rank=RANK, shuffle=False, drop_last=False)
    return DataLoader(ds, batch_size=args.batch_size, sampler=sampler, num_workers=2, pin_memory=True, collate_fn=collate)
'''

new_loader = '''def make_eval_loader(ds):
    return DataLoader(ds, batch_size=args.batch_size, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate)
'''

assert old_loader in script, "Evaluation loader section not found"
script = script.replace(old_loader, new_loader)

start = script.index("def evaluate(loader):")
end = script.index("def save_trainable(path):", start)

new_evaluate = '''def evaluate(loader):
    result = torch.zeros(3, dtype=torch.float64, device=device)
    if is_main:
        model.module.eval()
        loss_sum = 0.0
        y_true, y_pred = [], []
        with torch.no_grad():
            for ids, mask, labels in loader:
                ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
                logits = model.module(ids, mask)
                loss = criterion(logits, labels)
                loss_sum += loss.item() * labels.size(0)
                y_true.extend(labels.cpu().tolist())
                y_pred.extend(logits.argmax(-1).cpu().tolist())
        result[0] = f1_score(y_true, y_pred, labels=list(range(NUM_LABELS)), average="macro", zero_division=0)
        result[1] = accuracy_score(y_true, y_pred)
        result[2] = loss_sum / max(1, len(loader.dataset))
    dist.broadcast(result, src=0)
    return tuple(result.cpu().tolist())

'''

script = script[:start] + new_evaluate + script[end:]

with open(path, "w", encoding="utf-8") as f:
    f.write(script)

print("Patched evaluation successfully.")
print("Validation now uses the full datasets on rank 0, without duplicate samples or FP16.")

Patched evaluation successfully.
Validation now uses the full datasets on rank 0, without duplicate samples or FP16.


In [46]:
path = "/kaggle/working/train_rr_m2_lora.py"

with open(path, "r", encoding="utf-8") as f:
    script = f.read()

script = script.replace(
    'p.add_argument("--batch-size", type=int, default=16)',
    'p.add_argument("--batch-size", type=int, default=16)\n    p.add_argument("--max-steps", type=int, default=0)'
)

script = script.replace(
    '    for ids, mask, labels in iterator:\n',
    '    for step_idx, (ids, mask, labels) in enumerate(iterator):\n'
)

script = script.replace(
    '        if is_main:\n            iterator.set_postfix(loss=f"{running_loss/max(1,seen):.4f}")',
    '        if is_main:\n            iterator.set_postfix(loss=f"{running_loss/max(1,seen):.4f}")\n        if args.max_steps > 0 and step_idx + 1 >= args.max_steps:\n            break'
)

with open(path, "w", encoding="utf-8") as f:
    f.write(script)

print("Diagnostic option added.")

Diagnostic option added.


In [51]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_rr_m2_lora.py --epochs 5 --batch-size 16

W1016 14:22:39.085000 745 torch/distributed/run.py:851] 
W1016 14:22:39.085000 745 torch/distributed/run.py:851] *****************************************
W1016 14:22:39.085000 745 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1016 14:22:39.085000 745 torch/distributed/run.py:851] *****************************************
Parameters: 96,367,245; trainable: 579,725 (0.60%)
Epoch 1/5:   0%|                                        | 0/527 [00:00<?, ?it/s]/kaggle/working/train_rr_m2_lora.py:118: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.as_tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)
/kaggle/working/train_rr_m

In [56]:
import os

roots = [
    "/kaggle/input",
    "/root/.cache/huggingface",
    "/kaggle/working",
]

matches = []

for root in roots:
    if not os.path.isdir(root):
        continue

    for current, dirs, files in os.walk(root):
        # Skip large, irrelevant directories where possible
        dirs[:] = [
            d for d in dirs
            if d not in [".git", "__pycache__"]
        ]

        for filename in files:
            lower = filename.lower()

            if (
                lower in ["cl_test.json", "it_test.json"]
                or (
                    lower.endswith(".json")
                    and "phase_3_rr" in current.lower()
                    and "/test" in current.lower()
                )
            ):
                matches.append(os.path.join(current, filename))

print("Matching test files:")
for path in sorted(set(matches)):
    print(path)

if not matches:
    print("\nNo test JSON files found in the searched locations.")
    print("We will need to add the Phase_3_RR dataset as an input to this notebook.")

Matching test files:
/kaggle/working/phase3_data/Phase_3_RR/Test/CL_test.json
/kaggle/working/phase3_data/Phase_3_RR/Test/IT_test.json


In [58]:
import os
import json
import pickle
import torch
from tqdm.auto import tqdm

OUT = "/kaggle/working/phase3_rr/experiment_2_m2_lora"
TEST_DIR = "/kaggle/working/phase3_data/Phase_3_RR/Test"
PRED_PATH = os.path.join(OUT, "prediction.json")

MAX_LEN = int(cfg["max_len"])
PAD_ID = int(cfg["pad_id"])
BATCH_SIZE = 32

# Load the tokenizer saved by the M2 notebook
with open(os.path.join(OUT, "tokenizer.pkl"), "rb") as f:
    tokenizer = pickle.load(f)

# ID-to-label mapping
id_to_label = {v: k for k, v in label_vocab.items()}

def encode_sentence(text):
    ids = tokenizer.encode_text(str(text))

    # Match the training-time truncation and EOS handling
    ids = list(ids)
    ids = ids[:MAX_LEN - 1] + [tokenizer.eos_id]

    if len(ids) == 0:
        ids = [tokenizer.eos_id]

    return ids

def predict_document_sentences(sentences):
    predictions = []

    for start in tqdm(
        range(0, len(sentences), BATCH_SIZE),
        desc="Predicting sentences",
        leave=False
    ):
        batch_sentences = sentences[start:start + BATCH_SIZE]
        encoded = [encode_sentence(s) for s in batch_sentences]

        max_batch_len = max(len(x) for x in encoded)
        input_ids = torch.full(
            (len(encoded), max_batch_len),
            PAD_ID,
            dtype=torch.long
        )

        attention_mask = torch.zeros_like(input_ids)

        for i, ids in enumerate(encoded):
            input_ids[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
            attention_mask[i, :len(ids)] = 1

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        with torch.inference_mode():
            logits = model(input_ids, attention_mask)
            pred_ids = logits.argmax(dim=-1).cpu().tolist()

        predictions.extend(id_to_label[i] for i in pred_ids)

    return predictions

# Generate predictions for each domain
all_predictions = {}

for domain in ["CL", "IT"]:
    test_path = os.path.join(TEST_DIR, f"{domain}_test.json")
    assert os.path.isfile(test_path), f"Missing test file: {test_path}"

    with open(test_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)

    print(f"\n{domain}: {len(test_data)} documents")

    for doc_id, sentences in test_data.items():
        all_predictions[doc_id] = predict_document_sentences(sentences)

        assert len(all_predictions[doc_id]) == len(sentences), (
            f"Sentence count mismatch for {doc_id}"
        )

# Save predictions
with open(PRED_PATH, "w", encoding="utf-8") as f:
    json.dump(all_predictions, f, ensure_ascii=False, indent=2)

print("\nSaved predictions to:", PRED_PATH)
print("Total documents:", len(all_predictions))
print("Total sentence predictions:", sum(len(v) for v in all_predictions.values()))


CL: 5 documents


Predicting sentences:   0%|          | 0/1 [00:00<?, ?it/s]

Predicting sentences:   0%|          | 0/8 [00:00<?, ?it/s]

Predicting sentences:   0%|          | 0/2 [00:00<?, ?it/s]

Predicting sentences:   0%|          | 0/4 [00:00<?, ?it/s]

Predicting sentences:   0%|          | 0/30 [00:00<?, ?it/s]

... (3 outputs left out)

Predicting sentences:   0%|          | 0/2 [00:00<?, ?it/s]

Predicting sentences:   0%|          | 0/2 [00:00<?, ?it/s]

Predicting sentences:   0%|          | 0/5 [00:00<?, ?it/s]


Saved predictions to: /kaggle/working/phase3_rr/experiment_2_m2_lora/prediction.json
Total documents: 10
Total sentence predictions: 2197


In [59]:
import os
import json
import zipfile
from collections import Counter

OUT = "/kaggle/working/phase3_rr/experiment_2_m2_lora"
TEST_DIR = "/kaggle/working/phase3_data/Phase_3_RR/Test"

PRED_PATH = os.path.join(OUT, "prediction.json")
ZIP_PATH = "/kaggle/working/submission.zip"

# Load predictions and label vocabulary
with open(PRED_PATH, "r", encoding="utf-8") as f:
    predictions = json.load(f)

with open(os.path.join(OUT, "label_vocab.json"), "r", encoding="utf-8") as f:
    label_vocab = json.load(f)

valid_labels = set(label_vocab.keys())
expected_docs = {}

# Load the original test data to validate IDs and sentence counts
for domain in ["CL", "IT"]:
    test_path = os.path.join(TEST_DIR, f"{domain}_test.json")

    with open(test_path, "r", encoding="utf-8") as f:
        domain_data = json.load(f)

    expected_docs.update(domain_data)

assert set(predictions.keys()) == set(expected_docs.keys()), (
    "Document IDs do not match the test data."
)

for doc_id, sentences in expected_docs.items():
    pred = predictions[doc_id]

    assert isinstance(pred, list), f"{doc_id}: predictions must be a list"
    assert len(pred) == len(sentences), (
        f"{doc_id}: expected {len(sentences)} predictions, got {len(pred)}"
    )

    invalid = set(pred) - valid_labels
    assert not invalid, f"{doc_id}: invalid labels {invalid}"

total_sentences = sum(len(v) for v in predictions.values())
label_counts = Counter(
    label
    for document_predictions in predictions.values()
    for label in document_predictions
)

print("Validation passed.")
print("Documents:", len(predictions))
print("Total sentence predictions:", total_sentences)
print("All document IDs match:", True)
print("All sentence counts match:", True)
print("All labels valid:", True)
print("\nPrediction label distribution:")
for label, count in label_counts.most_common():
    print(f"{label}: {count}")

# Create submission ZIP with prediction.json at the ZIP root
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(PRED_PATH, arcname="prediction.json")

# Verify ZIP contents
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    contents = zf.namelist()
    assert contents == ["prediction.json"], f"Unexpected ZIP contents: {contents}"

    zipped_predictions = json.loads(
        zf.read("prediction.json").decode("utf-8")
    )
    assert zipped_predictions == predictions

print("\nZIP verification passed.")
print("ZIP contents:", contents)
print("Submission ZIP:", ZIP_PATH)
print("ZIP size (KB):", round(os.path.getsize(ZIP_PATH) / 1024, 2))

Validation passed.
Documents: 10
Total sentence predictions: 2197
All document IDs match: True
All sentence counts match: True
All labels valid: True

Prediction label distribution:
Fact: 404
PrecedentReliedUpon: 342
RatioOfTheDecision: 302
ArgumentPetitioner: 300
Statute: 245
RulingByLowerCourt: 169
RulingByPresentCourt: 103
PrecedentNotReliedUpon: 98
None: 96
ArgumentRespondent: 89
Dissent: 38
Issue: 11

ZIP verification passed.
ZIP contents: ['prediction.json']
Submission ZIP: /kaggle/working/submission.zip
ZIP size (KB): 2.51


PHASE 4

In [2]:
import os
import json
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE4_FILES = {
    "train": "Phase_4_CJPE/Train/train.json",
    "dev": "Phase_4_CJPE/Train/dev.json",
    "test": "Phase_4_CJPE/Test/test.json",
}
phase4_paths = {}
for split, filename in PHASE4_FILES.items():
    print(f"Accessing {split}: {filename}")
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    phase4_paths[split] = path
    print("Cached path:", path)

with open(phase4_paths["train"], "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(phase4_paths["dev"], "r", encoding="utf-8") as f:
    dev_data = json.load(f)

with open(phase4_paths["test"], "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("\nDataset sizes:")
print("Train:", len(train_data))
print("Dev:", len(dev_data))
print("Test:", len(test_data))
print("\nFirst training example keys:", train_data[0].keys())
print("First test example keys:", test_data[0].keys())

Accessing train: Phase_4_CJPE/Train/train.json


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Cached path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/train.json
Accessing dev: Phase_4_CJPE/Train/dev.json


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Cached path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/dev.json
Accessing test: Phase_4_CJPE/Test/test.json


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Cached path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Test/test.json

Dataset sizes:
Train: 37387
Dev: 1988
Test: 1517

First training example keys: dict_keys(['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5'])
First test example keys: dict_keys(['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5'])


In [7]:
import torch
from m2_model import M2Model

ckpt_path = "/kaggle/working/m2_checkpoints/latest.pt"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

model = M2Model(
    vocab_size=32000,
    d_model=640,
    n_layers=18,
    n_heads=10,
    d_ff=2560,
    dropout=0.05
)

result = model.load_state_dict(ckpt["model"], strict=True)

print("Phase 1 weights loaded successfully")
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Checkpoint step:", ckpt["step"])

Phase 1 weights loaded successfully
Parameters: 95787520
Checkpoint step: 30518


In [8]:
import torch
import torch.nn as nn
from m2_model import M2Model

class M2Classifier(nn.Module):
    def __init__(self, pretrained_model):
        super().__init__()

        self.backbone = pretrained_model
        self.classifier = nn.Linear(640, 2)

        # Freeze all pretrained parameters first
        for param in self.backbone.parameters():
            param.requires_grad = False

        # BitFit: unfreeze existing bias parameters
        for name, param in self.backbone.named_parameters():
            if name.endswith(".bias"):
                param.requires_grad = True

    def forward(self, input_ids, attention_mask):
        x = self.backbone.token_emb(input_ids)

        for block in self.backbone.blocks:
            x = block(x)

        x = self.backbone.final_ln(x)

        # Masked mean pooling
        mask = attention_mask.unsqueeze(-1).float()
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

        return self.classifier(x)


ckpt = torch.load(
    "/kaggle/working/m2_checkpoints/latest.pt",
    map_location="cpu",
    weights_only=False
)

backbone = M2Model(
    vocab_size=32000,
    d_model=640,
    n_layers=18,
    n_heads=10,
    d_ff=2560,
    dropout=0.05
)
backbone.load_state_dict(ckpt["model"], strict=True)

model = M2Classifier(backbone)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Trainable percentage: {100 * trainable / total:.2f}%")

Total parameters: 95,788,802
Trainable parameters: 82,562
Trainable percentage: 0.09%


In [12]:
import pickle
import torch
from tqdm.auto import tqdm

# Load the 32K tokenizer
with open("/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl", "rb") as f:
    tokenizer = pickle.load(f)

# Remove duplicate training IDs
seen = set()
train_unique = []

for row in train_data:
    if row["id"] not in seen:
        seen.add(row["id"])
        train_unique.append(row)

print("Unique train examples:", len(train_unique))

def encode_rows(rows, name):
    ids_list = []
    labels = []
    case_ids = []

    for row in tqdm(rows, desc=name):
        ids = tokenizer.encode_text(row["text"])[:1024]
        ids_list.append(torch.tensor(ids, dtype=torch.long))
        labels.append(int(row["label"]))
        case_ids.append(row["id"])

    return {
        "ids": ids_list,
        "labels": torch.tensor(labels, dtype=torch.long),
        "case_ids": case_ids
    }

train_enc = encode_rows(train_unique, "Tokenizing train")
dev_enc = encode_rows(dev_data, "Tokenizing dev")

torch.save(train_enc, "/kaggle/working/cjpe_m2_train.pt")
torch.save(dev_enc, "/kaggle/working/cjpe_m2_dev.pt")

print("Saved tokenized datasets.")
print("Train:", len(train_enc["ids"]), "Dev:", len(dev_enc["ids"]))
print("Example token length:", len(train_enc["ids"][0]))

Unique train examples: 32305


Tokenizing train:   0%|          | 0/32305 [00:00<?, ?it/s]

Tokenizing dev:   0%|          | 0/1988 [00:00<?, ?it/s]

Saved tokenized datasets.
Train: 32305 Dev: 1988
Example token length: 1024


In [25]:

%%writefile /kaggle/working/train_m2_cjpe.py
import os
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from sklearn.metrics import f1_score, precision_score, recall_score

from m2_model import M2Model


# ---------------- CONFIG ----------------
SEED = 42
MAX_LEN = 256
BATCH_SIZE = 2
EPOCHS = 3
LR = 1e-4
NUM_WORKERS = 2

PRETRAINED = "/kaggle/working/m2_checkpoints/latest.pt"
TRAIN_FILE = "/kaggle/working/cjpe_m2_train.pt"
DEV_FILE = "/kaggle/working/cjpe_m2_dev.pt"
OUT_DIR = "/kaggle/working/cjpe_m2"

os.makedirs(OUT_DIR, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class CJDataset(Dataset):
    def __init__(self, path):
        self.data = torch.load(path, map_location="cpu",
                                weights_only=False)

    def __len__(self):
        return len(self.data["labels"])

    def __getitem__(self, idx):
        # Truncate before batching to reduce transformer computation.
        ids = self.data["ids"][idx][:MAX_LEN]
        label = int(self.data["labels"][idx])
        return torch.as_tensor(ids, dtype=torch.long), label


def collate_fn(batch):
    ids = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch],
                          dtype=torch.long)

    lengths = torch.tensor([len(x) for x in ids],
                           dtype=torch.long)

    input_ids = nn.utils.rnn.pad_sequence(
        ids, batch_first=True, padding_value=0
    )

    # Use lengths, NOT input_ids != 0:
    # token ID 0 is also the tokenizer's EOS token.
    positions = torch.arange(input_ids.size(1)).unsqueeze(0)
    mask = positions < lengths.unsqueeze(1)

    return input_ids, mask, labels


class M2Classifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(640, 2)

        # Freeze the pretrained backbone.
        for p in self.backbone.parameters():
            p.requires_grad = False

        # BitFit: train existing bias parameters only.
        for name, p in self.backbone.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True

    def forward(self, input_ids, mask):
        # Extract hidden states directly, before lm_head.
        x = self.backbone.token_emb(input_ids)

        for block in self.backbone.blocks:
            x = block(x)

        hidden = self.backbone.final_ln(x)
        # hidden shape: [batch, seq_len, 640]

        # Masked mean pooling over non-padding tokens.
        mask_float = mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask_float).sum(dim=1)
        pooled = pooled / mask_float.sum(dim=1).clamp(min=1.0)

        # Binary classification logits: [batch, 2]
        return self.classifier(pooled)


def main():
    set_seed(SEED)

    local_rank = int(os.environ["LOCAL_RANK"])
    rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl")
    device = torch.device("cuda", local_rank)

    try:
        train_ds = CJDataset(TRAIN_FILE)
        dev_ds = CJDataset(DEV_FILE)

        train_sampler = DistributedSampler(
            train_ds, num_replicas=world_size,
            rank=rank, shuffle=True, seed=SEED
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=BATCH_SIZE,
            sampler=train_sampler,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            collate_fn=collate_fn,
            persistent_workers=(NUM_WORKERS > 0)
        )

        dev_loader = None
        if rank == 0:
            dev_loader = DataLoader(
                dev_ds,
                batch_size=4,
                shuffle=False,
                num_workers=NUM_WORKERS,
                pin_memory=True,
                collate_fn=collate_fn,
                persistent_workers=(NUM_WORKERS > 0)
            )

        checkpoint = torch.load(
            PRETRAINED, map_location="cpu", weights_only=False
        )

        backbone = M2Model(
            vocab_size=32000,
            d_model=640,
            n_layers=18,
            n_heads=10,
            d_ff=2560,
            dropout=0.05
        )

        state = checkpoint["model"] if "model" in checkpoint else checkpoint
        backbone.load_state_dict(state, strict=True)

        model = M2Classifier(backbone).to(device)

        trainable = sum(
            p.numel() for p in model.parameters() if p.requires_grad
        )

        model = DDP(
            model,
            device_ids=[local_rank],
            output_device=local_rank
        )

        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR,
            weight_decay=0.01
        )

        criterion = nn.CrossEntropyLoss()
        scaler = torch.amp.GradScaler("cuda")

        if rank == 0:
            print("Train/dev:", len(train_ds), len(dev_ds))
            print("GPUs:", world_size, "| Batch/GPU:", BATCH_SIZE)
            print("Sequence length:", MAX_LEN)
            print("Trainable parameters:", trainable)
            print("Running smoke test...", flush=True)

        # -------- SMOKE TEST: forward + backward only --------
        model.train()
        batch = next(iter(train_loader))
        x, mask, y = [t.to(device, non_blocking=True) for t in batch]

        if x.min().item() < 0 or x.max().item() >= 32000:
            raise ValueError("Token IDs outside vocabulary range.")

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(x, mask)
            loss = criterion(logits, y)

        if not torch.isfinite(loss).item():
            raise RuntimeError("Smoke-test loss is not finite.")

        scaler.scale(loss).backward()
        has_grad = False
        for p in model.parameters():
            if p.requires_grad and p.grad is not None:
                if not torch.isfinite(p.grad).all().item():
                    raise RuntimeError("Non-finite gradient in smoke test.")
                has_grad = True
        
        if not has_grad:
            raise RuntimeError("No trainable gradients in smoke test.")
        
        optimizer.zero_grad(set_to_none=True)
        del x, mask, y, logits, loss, batch
        torch.cuda.empty_cache()
        dist.barrier()

        if rank == 0:
            print("SMOKE TEST PASSED.", flush=True)

        best_f1 = -1.0

        for epoch in range(EPOCHS):
            train_sampler.set_epoch(epoch)
            model.train()
            total_loss = 0.0
            steps = 0

            progress = tqdm(
                train_loader,
                desc="Epoch {}/{}".format(epoch + 1, EPOCHS),
                disable=(rank != 0)
            )

            for x, mask, y in progress:
                x = x.to(device, non_blocking=True)
                mask = mask.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.autocast(
                    device_type="cuda", dtype=torch.float16
                ):
                    logits = model(x, mask)
                    loss = criterion(logits, y)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    1.0
                )
                scaler.step(optimizer)
                scaler.update()

                total_loss += loss.item()
                steps += 1

                if rank == 0:
                    progress.set_postfix(loss=round(loss.item(), 4))

            dist.barrier()

            if rank == 0:
                model.eval()
                all_preds = []
                all_labels = []
                dev_loss = 0.0
                dev_steps = 0

                # Evaluate through the underlying module on rank 0 only.
                with torch.inference_mode():
                    for x, mask, y in tqdm(
                        dev_loader, desc="Validation"
                    ):
                        x = x.to(device, non_blocking=True)
                        mask = mask.to(device, non_blocking=True)
                        y = y.to(device, non_blocking=True)

                        with torch.autocast(
                            device_type="cuda", dtype=torch.float16
                        ):
                            logits = model.module(x, mask)
                            loss = criterion(logits, y)

                        dev_loss += loss.item()
                        dev_steps += 1
                        all_preds.extend(logits.argmax(1).cpu().tolist())
                        all_labels.extend(y.cpu().tolist())

                macro_f1 = f1_score(
                    all_labels, all_preds, average="macro",
                    zero_division=0
                )
                precision = precision_score(
                    all_labels, all_preds, zero_division=0
                )
                recall = recall_score(
                    all_labels, all_preds, zero_division=0
                )

                print(
                    "Epoch {} | train_loss={:.4f} | dev_loss={:.4f} "
                    "| macro_f1={:.4f} | precision={:.4f} | recall={:.4f}"
                    .format(
                        epoch + 1,
                        total_loss / max(steps, 1),
                        dev_loss / max(dev_steps, 1),
                        macro_f1, precision, recall
                    ),
                    flush=True
                )

                saved = {
                    "epoch": epoch + 1,
                    "model": model.module.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "macro_f1": macro_f1
                }

                torch.save(saved, OUT_DIR + "/latest.pt")

                if macro_f1 > best_f1:
                    best_f1 = macro_f1
                    torch.save(saved, OUT_DIR + "/best.pt")
                    print("Saved best checkpoint.", flush=True)

            dist.barrier()

    finally:
        dist.destroy_process_group()

if __name__ == "__main__":
    main()


Overwriting /kaggle/working/train_m2_cjpe.py


In [26]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_m2_cjpe.py

W1021 17:03:19.866000 424 torch/distributed/run.py:851] 
W1021 17:03:19.866000 424 torch/distributed/run.py:851] *****************************************
W1021 17:03:19.866000 424 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1021 17:03:19.866000 424 torch/distributed/run.py:851] *****************************************
Train/dev: 32305 1988
GPUs: 2 | Batch/GPU: 2
Sequence length: 256
Trainable parameters: 82562
Running smoke test...
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
SMOKE TEST PASSED.
Validation: 100%|█████████████████████████████| 497/497 [00:09<00:00, 53.85it/s]
Epoch 1 | train_los

In [29]:

import os
import inspect
import torch

script = "/kaggle/working/train_m2_cjpe.py"
with open(script, "r") as f:
    lines = f.readlines()

print("=== M2 classifier and model definitions ===")
for i, line in enumerate(lines):
    if "class M2Classifier" in line or "def forward(" in line:
        print(f"\n--- around line {i+1} ---")
        print("".join(lines[i:min(i+24, len(lines))]))

print("\n=== M2 checkpoint ===")
ckpt_path = "/kaggle/working/cjpe_m2/best.pt"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print("Keys:", list(ckpt.keys()))
print("Epoch:", ckpt.get("epoch"))
print("Macro-F1:", ckpt.get("macro_f1", ckpt.get("best_f1")))

print("\n=== Tokenizer ===")
print("Class:", type(tokenizer).__name__)
print("encode_text:", inspect.signature(tokenizer.encode_text))

# Use the same maximum sequence length as M2 training.
MAX_LEN_M2 = 256
out_dir = "/kaggle/working/cjpe_test_tokens"
os.makedirs(out_dir, exist_ok=True)

test_ids_m2 = []
test_case_ids_m2 = []

for row in test_data:
    ids = tokenizer.encode_text(row["text"], add_eos=True)
    ids = ids[:MAX_LEN_M2]
    if len(ids) == 0:
        ids = [0]
    test_ids_m2.append(torch.tensor(ids, dtype=torch.long))
    test_case_ids_m2.append(str(row["id"]))

test_path_m2 = os.path.join(out_dir, "test_m2.pt")
torch.save({"ids": test_ids_m2, "case_ids": test_case_ids_m2}, test_path_m2)

print("\nSaved test tokens:", test_path_m2)
print("Cases:", len(test_ids_m2))
print("Max length:", max(len(x) for x in test_ids_m2))


=== M2 classifier and model definitions ===

--- around line 74 ---
class M2Classifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(640, 2)

        # Freeze the pretrained backbone.
        for p in self.backbone.parameters():
            p.requires_grad = False

        # BitFit: train existing bias parameters only.
        for name, p in self.backbone.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True

    def forward(self, input_ids, mask):
        # Extract hidden states directly, before lm_head.
        x = self.backbone.token_emb(input_ids)

        for block in self.backbone.blocks:
            x = block(x)

        hidden = self.backbone.final_ln(x)
        # hidden shape: [batch, seq_len, 640]


--- around line 89 ---
    def forward(self, input_ids, mask):
        # Extract hidden states directly, before lm_head.
        x = self.bac

In [30]:

import os
import json
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm
import torch.nn as nn
import importlib.util

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the exact M2 model definitions used for training.
SCRIPT = "/kaggle/working/train_m2_cjpe.py"
spec = importlib.util.spec_from_file_location("m2_cjpe_train", SCRIPT)
m2_train = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m2_train)

CKPT_PATH = "/kaggle/working/cjpe_m2/best.pt"
TEST_PATH = "/kaggle/working/cjpe_test_tokens/test_m2.pt"
OUT_PATH = "/kaggle/working/submission_m2_p4.json"

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
print("Checkpoint epoch:", ckpt["epoch"])
print("Validation Macro-F1:", ckpt["macro_f1"])

# Infer backbone architecture from the training script's own constructor.
# Reuse the model object already created in the notebook if it is the
# trained architecture; checkpoint loading below verifies compatibility.
model = globals().get("model", None)
if model is None:
    raise RuntimeError("No M2 model variable found. Please inspect the training script constructor.")

model.load_state_dict(ckpt["model"], strict=True)
model.to(device).eval()

test = torch.load(TEST_PATH, map_location="cpu", weights_only=False)
assert len(test["ids"]) == len(test["case_ids"]) == 1517

class M2TestDataset(Dataset):
    def __len__(self):
        return len(test["ids"])

    def __getitem__(self, idx):
        return test["ids"][idx], test["case_ids"][idx]

def m2_collate(batch):
    sequences = [x[0] for x in batch]
    lengths = torch.tensor([len(x) for x in sequences], dtype=torch.long)
    input_ids = pad_sequence(sequences, batch_first=True, padding_value=0)
    mask = torch.arange(input_ids.size(1)).unsqueeze(0) < lengths.unsqueeze(1)
    return input_ids, mask, [x[1] for x in batch]

loader = DataLoader(
    M2TestDataset(), batch_size=4, shuffle=False, collate_fn=m2_collate
)

predictions = []
with torch.no_grad():
    for input_ids, mask, case_ids in tqdm(loader, desc="M2 predictions"):
        logits = model(input_ids.to(device), mask.to(device))
        preds = logits.argmax(dim=1).cpu().tolist()
        predictions.extend(
            {"id": case_id, "pred": int(pred)}
            for case_id, pred in zip(case_ids, preds)
        )

assert len(predictions) == 1517
assert all(p["pred"] in (0, 1) for p in predictions)

with open(OUT_PATH, "w") as f:
    json.dump(predictions, f)

print("Saved:", OUT_PATH)
print("Total:", len(predictions))
print("Class counts:", {
    0: sum(p["pred"] == 0 for p in predictions),
    1: sum(p["pred"] == 1 for p in predictions)
})
print("Preview:", predictions[:3])


Checkpoint epoch: 2
Validation Macro-F1: 0.3708034565177422


M2 predictions:   0%|          | 0/380 [00:00<?, ?it/s]

Saved: /kaggle/working/submission_m2_p4.json
Total: 1517
Class counts: {0: 1486, 1: 31}
Preview: [{'id': '1986_397', 'pred': 0}, {'id': '1977_183', 'pred': 0}, {'id': '1993_98', 'pred': 0}]


In [31]:
import os
import json
import zipfile
import glob
import shutil

WORK = "/kaggle/working"

# Known M1 submission and possible names for M2/M3.
candidates = {"M2": [f"{WORK}/submission_m2_p4.json"]}

for model_name, paths in candidates.items():
    existing = [p for p in paths if os.path.isfile(p)]

    if not existing:
        print(f"{model_name}: prediction JSON not found yet.")
        continue

    source = existing[0]

    with open(source, "r") as f:
        predictions = json.load(f)

    assert isinstance(predictions, list), (
        f"{model_name}: expected a JSON list"
    )
    assert all(
        isinstance(row, dict)
        and "id" in row
        and "pred" in row
        and row["pred"] in (0, 1)
        for row in predictions
    ), f"{model_name}: invalid prediction format"

    zip_path = f"{WORK}/submission_{model_name.lower()}_p4.zip"

    with zipfile.ZipFile(
        zip_path, "w", zipfile.ZIP_DEFLATED
    ) as zf:
        zf.write(source, arcname="prediction.json")

    print(
        f"{model_name}: {len(predictions)} predictions -> "
        f"{zip_path}"
    )

M2: 1517 predictions -> /kaggle/working/submission_m2_p4.zip


Phase 5

In [5]:
import os
import torch

MODEL_CODE_PATH = "/kaggle/working/m2_model.py"
CHECKPOINT_PATH = "/kaggle/working/m2_checkpoints/best.pt"

with open(MODEL_CODE_PATH, "r", encoding="utf-8") as f:
    exec(compile(f.read(), MODEL_CODE_PATH, "exec"), globals())

ckpt = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

backbone = M2Model(
    vocab_size=32000,
    d_model=640,
    n_layers=18,
    n_heads=10,
    d_ff=2560,
    dropout=0.05
)

result = backbone.load_state_dict(ckpt["model"], strict=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone = backbone.to(device)

print("Phase 1 M2 checkpoint loaded:", result)
print("Checkpoint step:", ckpt.get("step"))
print("Best validation loss:", ckpt.get("val_loss", ckpt.get("best_val")))
print("Device:", device)

Phase 1 M2 checkpoint loaded: <All keys matched successfully>
Checkpoint step: 30518
Best validation loss: 1.4605953693389893
Device: cuda


In [6]:

import os
import json
from huggingface_hub import hf_hub_download

REPO_ID = "Exploration-Lab/CS-781-Capstone"
DATA_DIR = "/kaggle/working/lsi_m2"
os.makedirs(DATA_DIR, exist_ok=True)

def load_lsi_split(filename):
    path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename="Phase_5_LSI/" + filename,
        token=os.environ.get("HF_TOKEN"),
        local_dir=DATA_DIR
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_lsi_split("Train/train.json")
dev_data = load_lsi_split("Train/dev.json")
test_data = load_lsi_split("Test/test.json")

print("Train:", len(train_data))
print("Dev:", len(dev_data))
print("Test:", len(test_data))
print("Example fields:", train_data[0].keys())
print("Example label count:", len(train_data[0]["labels"]))


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Train: 42750
Dev: 10181
Test: 13019
Example fields: dict_keys(['id', 'text', 'labels'])
Example label count: 3


In [10]:

import os
import pickle

TOKENIZER_PATH = (
    "/kaggle/input/datasets/spikie/capstone-split/"
    "legal_bpe_32k.pkl"
)

assert os.path.isfile(TOKENIZER_PATH), (
    "Shared tokenizer file not found. Check the attached Kaggle dataset."
)

with open(TOKENIZER_PATH, "rb") as f:
    tok = pickle.load(f)

print("Loaded shared tokenizer:", type(tok).__name__)
print("Vocabulary size:", tok.vocab_size)
print("Test encoding:", tok.encode_text("A legal case", add_eos=True))


Loaded shared tokenizer: ByteLevelBPE
Vocabulary size: 32000
Test encoding: [66, 33, 1320, 33, 422, 0]


In [11]:

import torch
from tqdm.auto import tqdm

WORK = "/kaggle/working/lsi_m2"
CHUNK_SIZE = 1024
OVERLAP = 128
STRIDE = CHUNK_SIZE - OVERLAP
LABEL_IDS = sorted(set(
    int(label)
    for row in train_data
    for label in row["labels"]
))
label_to_idx = {label: i for i, label in enumerate(LABEL_IDS)}

def encode_document(text):
    token_ids = tok.encode_text(text, add_eos=True)
    if len(token_ids) == 0:
        token_ids = [0]

    chunks = []
    for start in range(0, len(token_ids), STRIDE):
        chunk = token_ids[start:start + CHUNK_SIZE]
        if len(chunk) > 0:
            chunks.append(torch.tensor(chunk, dtype=torch.int32))
        if start + CHUNK_SIZE >= len(token_ids):
            break

    return chunks

def encode_split(rows, has_labels, name):
    encoded = []
    for row in tqdm(rows, desc="Tokenizing " + name):
        item = {
            "id": str(row["id"]),
            "chunks": encode_document(row["text"])
        }
        if has_labels:
            item["labels"] = [
                int(label) for label in row["labels"]
            ]
        encoded.append(item)

    output_path = os.path.join(WORK, name + ".pt")
    torch.save(encoded, output_path)
    print("Saved:", output_path)
    return encoded

train_encoded = encode_split(train_data, True, "train")
dev_encoded = encode_split(dev_data, True, "dev")
test_encoded = encode_split(test_data, False, "test")

torch.save(LABEL_IDS, os.path.join(WORK, "label_ids.pt"))

print("Number of labels:", len(LABEL_IDS))
print("Chunks in first training document:", len(train_encoded[0]["chunks"]))
print("Maximum chunks in a training document:",
      max(len(row["chunks"]) for row in train_encoded))


Tokenizing train:   0%|          | 0/42750 [00:00<?, ?it/s]

Saved: /kaggle/working/lsi_m2/train.pt


Tokenizing dev:   0%|          | 0/10181 [00:00<?, ?it/s]

Saved: /kaggle/working/lsi_m2/dev.pt


Tokenizing test:   0%|          | 0/13019 [00:00<?, ?it/s]

Saved: /kaggle/working/lsi_m2/test.pt
Number of labels: 100
Chunks in first training document: 4
Maximum chunks in a training document: 232


In [12]:
import os
import json
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.multiprocessing as mp

from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from sklearn.metrics import f1_score

# Paths
WORK = "/kaggle/working/lsi_m2"
CHECKPOINT_PATH = "/kaggle/working/m2_checkpoints/best.pt"
TRAIN_PATH = os.path.join(WORK, "train.pt")
DEV_PATH = os.path.join(WORK, "dev.pt")
TEST_PATH = os.path.join(WORK, "test.pt")
LABEL_PATH = os.path.join(WORK, "label_ids.pt")

# Training
SEED = 42
EPOCHS = 5
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
MAX_TRAIN_CHUNKS = 2
CHUNK_SIZE = 1024
NUM_WORKERS = 2

# Evaluation
THRESHOLD = 0.5

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

assert os.path.isfile(CHECKPOINT_PATH), "M2 Phase 1 checkpoint not found."
assert os.path.isfile(TRAIN_PATH), "Tokenized train split not found."
assert os.path.isfile(DEV_PATH), "Tokenized dev split not found."
assert os.path.isfile(TEST_PATH), "Tokenized test split not found."
assert os.path.isfile(LABEL_PATH), "Label mapping not found."

LABEL_IDS = torch.load(LABEL_PATH, map_location="cpu", weights_only=False)
NUM_LABELS = len(LABEL_IDS)

print("Labels:", NUM_LABELS)
print("Epochs:", EPOCHS)
print("Training chunks per document:", MAX_TRAIN_CHUNKS)
print("Checkpoint:", CHECKPOINT_PATH)

Labels: 100
Epochs: 5
Training chunks per document: 2
Checkpoint: /kaggle/working/m2_checkpoints/best.pt


In [13]:
ckpt = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

backbone_check = M2Model(
    vocab_size=32000,
    d_model=640,
    n_layers=18,
    n_heads=10,
    d_ff=2560,
    dropout=0.05
)

backbone_check.load_state_dict(ckpt["model"], strict=True)

total_params = 0
bias_params = 0
bias_names = []

for name, param in backbone_check.named_parameters():
    total_params += param.numel()

    # BitFit: train bias parameters only in the pretrained backbone.
    if name.endswith(".bias"):
        bias_params += param.numel()
        bias_names.append(name)

print("Checkpoint step:", ckpt.get("step"))
print("Total backbone parameters:", f"{total_params:,}")
print("Bias parameters:", f"{bias_params:,}")
print("Number of bias tensors:", len(bias_names))
print("First bias names:", bias_names[:15])

assert len(bias_names) > 0, "No bias parameters found; inspect M2 parameter names."

Checkpoint step: 30518
Total backbone parameters: 95,787,520
Bias parameters: 81,280
Number of bias tensors: 73
First bias names: ['blocks.0.ln1.bias', 'blocks.0.ln2.bias', 'blocks.0.ffn.0.bias', 'blocks.0.ffn.2.bias', 'blocks.1.ln1.bias', 'blocks.1.ln2.bias', 'blocks.1.ffn.0.bias', 'blocks.1.ffn.2.bias', 'blocks.2.ln1.bias', 'blocks.2.ln2.bias', 'blocks.2.ffn.0.bias', 'blocks.2.ffn.2.bias', 'blocks.3.ln1.bias', 'blocks.3.ln2.bias', 'blocks.3.ffn.0.bias']


In [25]:

%%writefile /kaggle/working/train_m2_lsi_ddp.py

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.distributed as dist

from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score
from tqdm import tqdm

# ============================================================
# Configuration
# ============================================================

WORK = "/kaggle/working/lsi_m2"
MODEL_CODE_PATH = "/kaggle/working/m2_model.py"
CHECKPOINT_PATH = "/kaggle/working/m2_checkpoints/best.pt"
OUTPUT_DIR = os.path.join(WORK, "checkpoints")

EPOCHS = 2
BATCH_SIZE = 4             # documents per GPU
GRAD_ACCUM = 2
MAX_TRAIN_CHUNKS = 1
EVAL_CHUNK_BATCH = 2
LR = 2e-4
WEIGHT_DECAY = 0.01
THRESHOLD = 0.5
SEED = 42

# ============================================================
# Dataset
# ============================================================

class LSIDataset(Dataset):
    def __init__(self, path, labeled=True):
        self.rows = torch.load(
            path, map_location="cpu", weights_only=False
        )
        self.labeled = labeled

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        chunks = row["chunks"]

        if self.labeled:
            # One random chunk per document for this training example.
            # Sampling is repeated when the document is accessed again.
            chunk_idx = random.randrange(len(chunks))
            chunks = [chunks[chunk_idx]]

        return {
            "id": row["id"],
            "chunks": chunks,
            "labels": row.get("labels", [])
        }


def collate_train(batch):
    ids = []
    lengths = []
    targets = []

    for row in batch:
        ids.append(row["chunks"][0].long())
        lengths.append(len(row["chunks"][0]))

        target = torch.zeros(NUM_LABELS, dtype=torch.float32)
        for label in row["labels"]:
            label = int(label)
            if label in LABEL_TO_IDX:
                target[LABEL_TO_IDX[label]] = 1.0
        targets.append(target)

    padded = pad_sequence(ids, batch_first=True, padding_value=0)

    return {
        "input_ids": padded,
        "lengths": torch.tensor(lengths, dtype=torch.long),
        "targets": torch.stack(targets)
    }


# ============================================================
# Model
# ============================================================

class M2LSI(nn.Module):
    def __init__(self, backbone, num_labels):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(640, num_labels)

    def forward(self, input_ids, lengths):
        backbone = self.backbone

        x = backbone.token_emb(input_ids)

        for block in backbone.blocks:
            x = block(x)

        # Hidden states before the vocabulary prediction head.
        x = backbone.final_ln(x)

        positions = torch.arange(
            x.size(1), device=x.device
        ).unsqueeze(0)

        mask = (positions < lengths.unsqueeze(1)).unsqueeze(-1)
        mask = mask.to(x.dtype)

        pooled = (x * mask).sum(dim=1)
        pooled = pooled / lengths.clamp(min=1).unsqueeze(1)

        return self.classifier(pooled)


def load_backbone():
    namespace = {}

    with open(MODEL_CODE_PATH, "r", encoding="utf-8") as f:
        exec(compile(f.read(), MODEL_CODE_PATH, "exec"), namespace)

    M2Model = namespace["M2Model"]

    backbone = M2Model(
        vocab_size=32000,
        d_model=640,
        n_layers=18,
        n_heads=10,
        d_ff=2560,
        dropout=0.05
    )

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False
    )

    backbone.load_state_dict(checkpoint["model"], strict=True)
    return backbone


def configure_bitfit(model):
    for param in model.backbone.parameters():
        param.requires_grad = False

    for name, param in model.backbone.named_parameters():
        if name.endswith(".bias"):
            param.requires_grad = True

    for param in model.classifier.parameters():
        param.requires_grad = True

    trainable = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    total = sum(p.numel() for p in model.parameters())

    for name, param in model.backbone.named_parameters():
        if param.requires_grad:
            assert name.endswith(".bias"), name

    print(
        f"Trainable parameters: {trainable:,} / {total:,}",
        flush=True
    )


# ============================================================
# Evaluation: one prediction per document
# ============================================================

@torch.no_grad()
def evaluate(model, rows, device):
    model.eval()

    backbone = model.backbone
    classifier = model.classifier

    all_predictions = []
    all_targets = []

    for row in tqdm(rows, desc="Evaluating", leave=False):
        chunks = row["chunks"]
        chunk_reps = []

        # Process every chunk, but only a small number at a time.
        for start in range(0, len(chunks), EVAL_CHUNK_BATCH):
            part = [
                chunk.long()
                for chunk in chunks[start:start + EVAL_CHUNK_BATCH]
            ]

            lengths = torch.tensor(
                [len(chunk) for chunk in part],
                dtype=torch.long,
                device=device
            )

            input_ids = pad_sequence(
                part, batch_first=True, padding_value=0
            ).to(device)

            x = backbone.token_emb(input_ids)

            for block in backbone.blocks:
                x = block(x)

            x = backbone.final_ln(x)

            positions = torch.arange(
                x.size(1), device=device
            ).unsqueeze(0)

            mask = (
                positions < lengths.unsqueeze(1)
            ).unsqueeze(-1).to(x.dtype)

            reps = (x * mask).sum(dim=1)
            reps = reps / lengths.clamp(min=1).unsqueeze(1)

            chunk_reps.append(reps)

        document_rep = torch.cat(chunk_reps, dim=0).mean(
            dim=0, keepdim=True
        )

        logits = classifier(document_rep)
        probabilities = torch.sigmoid(logits)

        all_predictions.append(probabilities.cpu())

        target = torch.zeros(NUM_LABELS, dtype=torch.float32)
        for label in row["labels"]:
            label = int(label)
            if label in LABEL_TO_IDX:
                target[LABEL_TO_IDX[label]] = 1.0

        all_targets.append(target.unsqueeze(0))

    probabilities = torch.cat(all_predictions, dim=0).numpy()
    targets = torch.cat(all_targets, dim=0).numpy().astype(np.int32)
    predictions = (probabilities >= THRESHOLD).astype(np.int32)

    return {
        "macro_f1": f1_score(
            targets, predictions, average="macro", zero_division=0
        ),
        "micro_f1": f1_score(
            targets, predictions, average="micro", zero_division=0
        ),
        "subset_accuracy": float(
            np.all(targets == predictions, axis=1).mean()
        )
    }


# ============================================================
# Distributed training
# ============================================================

def main():
    dist.init_process_group(backend="nccl")

    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ["LOCAL_RANK"])

    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    random.seed(SEED + rank)
    np.random.seed(SEED + rank)
    torch.manual_seed(SEED + rank)
    torch.cuda.manual_seed_all(SEED + rank)

    train_path = os.path.join(WORK, "train.pt")
    dev_path = os.path.join(WORK, "dev.pt")
    label_path = os.path.join(WORK, "label_ids.pt")

    global NUM_LABELS, LABEL_TO_IDX

    label_ids = torch.load(
        label_path, map_location="cpu", weights_only=False
    )
    LABEL_TO_IDX = {int(label): i for i, label in enumerate(label_ids)}
    NUM_LABELS = len(label_ids)

    train_dataset = LSIDataset(train_path, labeled=True)
    dev_rows = torch.load(
        dev_path, map_location="cpu", weights_only=False
    )

    sampler = DistributedSampler(
        train_dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        drop_last=True
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        collate_fn=collate_train,
        num_workers=0,
        pin_memory=True,
        drop_last=True
    )

    backbone = load_backbone()
    model = M2LSI(backbone, NUM_LABELS)
    configure_bitfit(model)
    model = model.to(device)

    model = DDP(
        model,
        device_ids=[local_rank],
        output_device=local_rank,
        find_unused_parameters=False
    )

    trainable_params = [
        p for p in model.parameters() if p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    criterion = nn.BCEWithLogitsLoss()

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    best_macro = -1.0

    for epoch in range(EPOCHS):
        sampler.set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)

        total_loss = 0.0
        number_batches = len(train_loader)

        iterator = train_loader
        if rank == 0:
            iterator = tqdm(
                train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"
            )

        for step, batch in enumerate(iterator):
            input_ids = batch["input_ids"].to(device)
            lengths = batch["lengths"].to(device)
            targets = batch["targets"].to(device)

            logits = model(input_ids, lengths)
            loss = criterion(logits, targets)
            total_loss += loss.item()

            (loss / GRAD_ACCUM).backward()

            if (
                (step + 1) % GRAD_ACCUM == 0
                or (step + 1) == number_batches
            ):
                torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

        loss_tensor = torch.tensor(
            total_loss / max(1, number_batches),
            dtype=torch.float64,
            device=device
        )
        dist.all_reduce(loss_tensor, op=dist.ReduceOp.SUM)
        mean_loss = loss_tensor.item() / world_size

        # Only rank 0 evaluates. All ranks synchronize after saving.
        if rank == 0:
            metrics = evaluate(model.module, dev_rows, device)

            print(
                f"Epoch {epoch + 1}/{EPOCHS} "
                f"loss={mean_loss:.4f} "
                f"Macro-F1={metrics['macro_f1']:.5f} "
                f"Micro-F1={metrics['micro_f1']:.5f} "
                f"Subset-Acc={metrics['subset_accuracy']:.5f}",
                flush=True
            )

            state = {
                "epoch": epoch + 1,
                "model": model.module.state_dict(),
                "optimizer": optimizer.state_dict(),
                "metrics": metrics,
                "label_ids": label_ids,
                "method": "BitFit",
                "architecture": "M2"
            }

            torch.save(
                state, os.path.join(OUTPUT_DIR, "m2_lsi_latest.pt")
            )

            if metrics["macro_f1"] > best_macro:
                best_macro = metrics["macro_f1"]
                torch.save(
                    state, os.path.join(OUTPUT_DIR, "m2_lsi_best.pt")
                )

        dist.barrier()

    dist.destroy_process_group()


if __name__ == "__main__":
    main()


Overwriting /kaggle/working/train_m2_lsi_ddp.py


In [29]:

# Patch the existing M2 DDP script for mixed precision.
# This keeps the current model, dataset, checkpoint, and training setup.

path = "/kaggle/working/train_m2_lsi_ddp.py"

with open(path, "r", encoding="utf-8") as f:
    script = f.read()

# 1. Add GradScaler after the optimizer if it isn't already present.
old_optimizer = '''    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )
'''
new_optimizer = old_optimizer + '''    scaler = torch.amp.GradScaler("cuda")
'''

if 'scaler = torch.amp.GradScaler("cuda")' not in script:
    if old_optimizer not in script:
        raise RuntimeError("Could not locate optimizer block; script unchanged.")
    script = script.replace(old_optimizer, new_optimizer, 1)

# 2. Replace FP32 training forward/backward with FP16 + scaling.
old_forward = '''            logits = model(input_ids, lengths)
            loss = criterion(logits, targets)
            total_loss += loss.item()

            (loss / GRAD_ACCUM).backward()
'''
new_forward = '''            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(input_ids, lengths)
                loss = criterion(logits, targets)

            total_loss += loss.item()
            scaler.scale(loss / GRAD_ACCUM).backward()
'''

if old_forward in script:
    script = script.replace(old_forward, new_forward, 1)
elif new_forward not in script:
    raise RuntimeError("Could not locate forward/backward block; script unchanged.")

# 3. Replace optimizer step with scaler-aware update.
old_step = '''                torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
'''
new_step = '''                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
'''

if old_step in script:
    script = script.replace(old_step, new_step, 1)
elif new_step not in script:
    raise RuntimeError("Could not locate optimizer step; script unchanged.")

with open(path, "w", encoding="utf-8") as f:
    f.write(script)

# Verify the important settings and edits.
with open(path, "r", encoding="utf-8") as f:
    updated = f.read()

assert "BATCH_SIZE = 8" in updated
assert 'torch.autocast("cuda", dtype=torch.float16)' in updated
assert 'torch.amp.GradScaler("cuda")' in updated
assert "scaler.scale(loss / GRAD_ACCUM).backward()" in updated
assert "scaler.step(optimizer)" in updated

print("Updated M2 DDP script successfully.")
print("Batch size per GPU: 8")
print("Gradient accumulation:", 1)
print("Mixed precision: FP16 with GradScaler")
print("Baseline training: disabled")


Updated M2 DDP script successfully.
Batch size per GPU: 8
Gradient accumulation: 1
Mixed precision: FP16 with GradScaler
Baseline training: disabled


In [30]:
!OMP_NUM_THREADS=1 torchrun --standalone --nproc_per_node=2 /kaggle/working/train_m2_lsi_ddp.py

Trainable parameters: 145,380 / 95,851,620
Trainable parameters: 145,380 / 95,851,620
Epoch 1/2:  43%|████████████                | 1151/2671 [08:46<11:29,  2.20it/s][rank1]:[E1028 14:04:45.368516311 ProcessGroupNCCL.cpp:2119] [PG ID 0 PG GUID 0(default_pg) Rank 1] Process group watchdog thread terminated with exception: CUDA error: an illegal memory access was encountered
Search for `cudaErrorIllegalAddress' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Exception raised from query at /pytorch/c10/cuda/CUDAEvent.h:111 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x9d (0x7c040b77205d in 

In [65]:
LOG = "/kaggle/working/m2_lsi_cuda_diagnostic.log"

with open(LOG, "r", errors="replace") as f:
    lines = f.readlines()

keywords = [
    "CUDA error",
    "illegal memory access",
    "Traceback",
    "[rank1]",
    "[rank0]",
    "Error",
    "RuntimeError",
    "Assertion",
    "timed out",
]

matches = [
    i for i, line in enumerate(lines)
    if any(k.lower() in line.lower() for k in keywords)
]

print("Total log lines:", len(lines))
print("Relevant sections:\n")

shown = set()
for i in matches:
    start = max(0, i - 4)
    end = min(len(lines), i + 12)

    if any(j in shown for j in range(start, end)):
        continue

    print(f"--- Log lines {start + 1}-{end} ---")
    print("".join(lines[start:end]))
    shown.update(range(start, end))

Total log lines: 107
Relevant sections:

--- Log lines 18-33 ---
Epoch 1/2:  42%|████▏     | 1119/2671 [01:00<04:49,  5.36it/s][rank1]:[E1028 15:02:24.746243713 ProcessGroupNCCL.cpp:2119] [PG ID 0 PG GUID 0(default_pg) Rank 1] Process group watchdog thread terminated with exception: CUDA error: an illegal memory access was encountered
Search for `cudaErrorIllegalAddress' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Exception raised from query at /pytorch/c10/cuda/CUDAEvent.h:111 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x9d (0x7f6ab657205d in /usr/local/lib/python3.13/dist-packages/torch/lib/libc10.so)
frame #1: <unknown function> + 0xc114 (0x7f6ae8d73114 in /usr/local/lib/python3.13/dist-packages/torch/lib/libc10_cuda.so)
frame #2: c10d::ProcessGroupN

Phase 6

In [6]:
import os
import json
import random
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILE = "Phase_6_Summ/Train/train.json"
TEST_FILE = "Phase_6_Summ/Test/test.json"

SEED = 42
random.seed(SEED)


train_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Training dataset:",train_path)

test_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

Training dataset: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Train/train.json


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

In [7]:
import os
import torch
import sys

MODEL_CODE_PATH = "/kaggle/working/m2_model.py"
CHECKPOINT_PATH = "/kaggle/working/m2_checkpoints/best.pt"

assert os.path.exists(MODEL_CODE_PATH)
assert os.path.exists(CHECKPOINT_PATH)

sys.path.insert(0,"/kaggle/working")
from m2_model import M2Model

ckpt = torch.load(CHECKPOINT_PATH,map_location="cpu",weights_only=False)

backbone = M2Model(
    vocab_size=32000,
    d_model=640,
    n_layers=18,
    n_heads=10,
    d_ff=2560,
    dropout=0.05
)

result = backbone.load_state_dict(ckpt["model"],strict=True)

print("Checkpoint loaded:",result)
print("Checkpoint keys:",list(ckpt.keys()))
print("Checkpoint step:",ckpt.get("step"))
print("Validation loss:",ckpt.get("val_loss",ckpt.get("best_val")))
print("Parameters:",f"{sum(p.numel() for p in backbone.parameters()):,}")
print("Attention:",type(backbone.blocks[0].attn).__name__)

Checkpoint loaded: <All keys matched successfully>
Checkpoint keys: ['step', 'model', 'val_loss', 'best_val']
Checkpoint step: 30518
Validation loss: 1.4605953693389893
Parameters: 95,787,520
Attention: MQA


In [8]:
import math
import torch
import torch.nn as nn

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

class LoRALinear(nn.Module):
    def __init__(self,base_layer,r=8,alpha=16,dropout=0.05):
        super().__init__()
        self.base = base_layer
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

        self.r = r
        self.scaling = alpha/r
        self.lora_dropout = nn.Dropout(dropout)
        self.lora_A = nn.Parameter(torch.empty(r,base_layer.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base_layer.out_features,r))
        nn.init.kaiming_uniform_(self.lora_A,a=math.sqrt(5))

    def forward(self,x):
        return self.base(x)+(self.lora_dropout(x) @ self.lora_A.T @ self.lora_B.T)*self.scaling


for p in backbone.parameters():
    p.requires_grad = False

for block in backbone.blocks:
    for name in ["q_proj","k_proj","v_proj","out_proj"]:
        layer = getattr(block.attn,name)
        setattr(block.attn,name,LoRALinear(layer,LORA_R,LORA_ALPHA,LORA_DROPOUT))

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total = sum(p.numel() for p in backbone.parameters())

print("LoRA modules:",sum(isinstance(getattr(block.attn,name),LoRALinear) for block in backbone.blocks for name in ["q_proj","k_proj","v_proj","out_proj"]))
print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters: {total:,}")
print(f"Trainable percentage: {100*trainable/total:.3f}%")
print("Trainable tensors:",sum(p.requires_grad for p in backbone.parameters()))

LoRA modules: 72
Trainable parameters: 571,392
Total parameters: 96,358,912
Trainable percentage: 0.593%
Trainable tensors: 144


In [9]:
with open(train_path,"r",encoding="utf-8") as f:
    train_data = json.load(f)

with open(test_path,"r",encoding="utf-8") as f:
    test_data = json.load(f)

In [11]:

import os
import pickle
import re
import math
import random
from collections import Counter
from tqdm.auto import tqdm


TOKENIZER_PATH = "/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl"

with open(TOKENIZER_PATH, "rb") as f:
    bpe = pickle.load(f)

print("Tokenizer loaded:", type(bpe).__name__)
print("Vocabulary:", bpe.vocab_size)
print("Train documents:", len(train_data))
print("Test documents:", len(test_data))

SEED = 42
MAX_SEQ_LEN = 2048
MAX_SOURCE_LEN = 1400
MAX_TARGET_LEN = 500
CHUNK_OVERLAP = 128
EOS_ID = bpe.eos_id

random.seed(SEED)

PROMPT = (
    "Summarize the following legal judgment segment. "
    "Include only information supported by the text.\n\n"
    "JUDGMENT:\n"
)
SUMMARY_PREFIX = "\n\nSUMMARY:\n"

PROMPT_IDS = bpe.encode_text(PROMPT, add_eos=False)
SUMMARY_PREFIX_IDS = bpe.encode_text(SUMMARY_PREFIX, add_eos=False)
def split_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


def lexical_words(text):
    return set(re.findall(r"\b\w+\b", text.lower()))

def encode_document_chunks(text):
    units = []

    for sentence in split_sentences(text):
        ids = bpe.encode_text(sentence, add_eos=False)

        if not ids:
            continue

        words = lexical_words(sentence)

        if len(ids) <= MAX_SOURCE_LEN:
            units.append((ids, words))
        else:
            stride = MAX_SOURCE_LEN - CHUNK_OVERLAP

            for start in range(0, len(ids), stride):
                part = ids[start:start + MAX_SOURCE_LEN]
                if part:
                    units.append((part, words))
                if start + MAX_SOURCE_LEN >= len(ids):
                    break

    if not units:
        return [[EOS_ID]], [set()]

    chunks = []
    chunk_words = []
    start = 0

    while start < len(units):
        current_ids = []
        current_words = set()
        end = start

        while end < len(units):
            ids, words = units[end]

            if len(current_ids) + len(ids) > MAX_SOURCE_LEN:
                break

            current_ids.extend(ids)
            current_words.update(words)
            end += 1

        if end == start:
            current_ids = units[start][0][:MAX_SOURCE_LEN]
            current_words.update(units[start][1])
            end += 1

        chunks.append(current_ids)
        chunk_words.append(current_words)

        if end >= len(units):
            break

        next_start = end
        overlap_tokens = 0
        j = end - 1

        while j >= start and overlap_tokens + len(units[j][0]) <= CHUNK_OVERLAP:
            overlap_tokens += len(units[j][0])
            next_start = j
            j -= 1

        if next_start <= start:
            next_start = end

        start = next_start

    return chunks, chunk_words

def make_target_segments(sentences):
    segments = []
    current = []

    for sentence in sentences:
        ids = bpe.encode_text(sentence, add_eos=False)
        start = 0

        while start < len(ids):
            space = MAX_TARGET_LEN - 1 - len(current)

            if space == 0:
                segments.append(current + [EOS_ID])
                current = []
                space = MAX_TARGET_LEN - 1

            take = min(space, len(ids) - start)
            current.extend(ids[start:start + take])
            start += take

            if len(current) == MAX_TARGET_LEN - 1:
                segments.append(current + [EOS_ID])
                current = []

    if current:
        segments.append(current + [EOS_ID])

    return segments

def build_document_examples(row):
    chunks, chunk_words = encode_document_chunks(row["document"])
    summary_sentences = split_sentences(row["summary"])

    df = Counter()

    for words in chunk_words:
        for word in words:
            df[word] += 1

    n_chunks = len(chunks)

    idf = {
        word: math.log((n_chunks + 1) / (count + 1)) + 1
        for word, count in df.items()
    }

    assigned = [[] for _ in chunks]

    for sentence in summary_sentences:
        words = lexical_words(sentence)

        if not words:
            continue

        scores = [
            sum(idf.get(word, 1.0) for word in words & chunk_set)
            for chunk_set in chunk_words
        ]

        best_idx = max(range(n_chunks), key=lambda i: scores[i])
        assigned[best_idx].append(sentence)

    examples = []

    for chunk_idx, source_ids in enumerate(chunks):
        target_segments = make_target_segments(assigned[chunk_idx])

        for target_ids in target_segments:
            examples.append({
                "id": str(row["id"]),
                "chunk_idx": chunk_idx,
                "source_ids": source_ids,
                "target_ids": target_ids
            })

    return examples


doc_ids = sorted(set(str(row["id"]) for row in train_data))
random.Random(SEED).shuffle(doc_ids)

val_count = max(1, int(len(doc_ids) * 0.10))

valid_ids = set(doc_ids[:val_count])
train_ids = set(doc_ids[val_count:])

assert train_ids.isdisjoint(valid_ids)

print("Train documents:", len(train_ids))
print("Validation documents:", len(valid_ids))

train_examples = []
valid_examples = []

for row in tqdm(train_data, desc="Preparing training documents"):
    examples = build_document_examples(row)

    if str(row["id"]) in valid_ids:
        valid_examples.extend(examples)
    else:
        train_examples.extend(examples)


test_chunks = []

for row in tqdm(test_data, desc="Preparing test documents"):
    chunks, _ = encode_document_chunks(row["document"])

    for chunk_idx, source_ids in enumerate(chunks):
        test_chunks.append({
            "id": str(row["id"]),
            "chunk_idx": chunk_idx,
            "source_ids": source_ids
        })


print("\nPreprocessing complete")
print("Training examples:", len(train_examples))
print("Validation examples:", len(valid_examples))
print("Test chunks:", len(test_chunks))

sample = train_examples[0]

print("\nSample ID:", sample["id"])
print("Source length:", len(sample["source_ids"]))
print("Target length:", len(sample["target_ids"]))

max_length = max(
    len(PROMPT_IDS) + len(x["source_ids"]) +
    len(SUMMARY_PREFIX_IDS) + len(x["target_ids"])
    for x in train_examples + valid_examples
)

print("Maximum sequence length:", max_length)
print("Train/validation IDs disjoint:", train_ids.isdisjoint(valid_ids))


Tokenizer loaded: ByteLevelBPE
Vocabulary: 32000
Train documents: 7030
Test documents: 100
Train documents: 6327
Validation documents: 703


Preparing training documents:   0%|          | 0/7030 [00:00<?, ?it/s]

Preparing test documents:   0%|          | 0/100 [00:00<?, ?it/s]


Preprocessing complete
Training examples: 44937
Validation examples: 5017
Test chunks: 816

Sample ID: 6712
Source length: 1037
Target length: 316
Maximum sequence length: 1941
Train/validation IDs disjoint: True


In [10]:

import os
import pickle
import re
import math
import random
from collections import Counter
from tqdm.auto import tqdm


TOKENIZER_PATH = "/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl"

with open(TOKENIZER_PATH, "rb") as f:
    bpe = pickle.load(f)

print("Tokenizer loaded:", type(bpe).__name__)
print("Vocabulary:", bpe.vocab_size)
print("Train documents:", len(train_data))
print("Test documents:", len(test_data))

SEED = 42
MAX_SEQ_LEN = 2048
MAX_SOURCE_LEN = 1400
MAX_TARGET_LEN = 500
CHUNK_OVERLAP = 128
EOS_ID = bpe.eos_id

random.seed(SEED)

PROMPT = (
    "Summarize the following legal judgment segment. "
    "Include only information supported by the text.\n\n"
    "JUDGMENT:\n"
)
SUMMARY_PREFIX = "\n\nSUMMARY:\n"

PROMPT_IDS = bpe.encode_text(PROMPT, add_eos=False)
SUMMARY_PREFIX_IDS = bpe.encode_text(SUMMARY_PREFIX, add_eos=False)
def split_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


def lexical_words(text):
    return set(re.findall(r"\b\w+\b", text.lower()))

def encode_document_chunks(text):
    units = []

    for sentence in split_sentences(text):
        ids = bpe.encode_text(sentence, add_eos=False)

        if not ids:
            continue

        words = lexical_words(sentence)

        if len(ids) <= MAX_SOURCE_LEN:
            units.append((ids, words))
        else:
            stride = MAX_SOURCE_LEN - CHUNK_OVERLAP

            for start in range(0, len(ids), stride):
                part = ids[start:start + MAX_SOURCE_LEN]
                if part:
                    units.append((part, words))
                if start + MAX_SOURCE_LEN >= len(ids):
                    break

    if not units:
        return [[EOS_ID]], [set()]

    chunks = []
    chunk_words = []
    start = 0

    while start < len(units):
        current_ids = []
        current_words = set()
        end = start

        while end < len(units):
            ids, words = units[end]

            if len(current_ids) + len(ids) > MAX_SOURCE_LEN:
                break

            current_ids.extend(ids)
            current_words.update(words)
            end += 1

        if end == start:
            current_ids = units[start][0][:MAX_SOURCE_LEN]
            current_words.update(units[start][1])
            end += 1

        chunks.append(current_ids)
        chunk_words.append(current_words)

        if end >= len(units):
            break

        next_start = end
        overlap_tokens = 0
        j = end - 1

        while j >= start and overlap_tokens + len(units[j][0]) <= CHUNK_OVERLAP:
            overlap_tokens += len(units[j][0])
            next_start = j
            j -= 1

        if next_start <= start:
            next_start = end

        start = next_start

    return chunks, chunk_words

def make_target_segments(sentences):
    segments = []
    current = []

    for sentence in sentences:
        ids = bpe.encode_text(sentence, add_eos=False)
        start = 0

        while start < len(ids):
            space = MAX_TARGET_LEN - 1 - len(current)

            if space == 0:
                segments.append(current + [EOS_ID])
                current = []
                space = MAX_TARGET_LEN - 1

            take = min(space, len(ids) - start)
            current.extend(ids[start:start + take])
            start += take

            if len(current) == MAX_TARGET_LEN - 1:
                segments.append(current + [EOS_ID])
                current = []

    if current:
        segments.append(current + [EOS_ID])

    return segments

def build_document_examples(row):
    chunks, chunk_words = encode_document_chunks(row["document"])
    summary_sentences = split_sentences(row["summary"])

    df = Counter()

    for words in chunk_words:
        for word in words:
            df[word] += 1

    n_chunks = len(chunks)

    idf = {
        word: math.log((n_chunks + 1) / (count + 1)) + 1
        for word, count in df.items()
    }

    assigned = [[] for _ in chunks]

    for sentence in summary_sentences:
        words = lexical_words(sentence)

        if not words:
            continue

        scores = [
            sum(idf.get(word, 1.0) for word in words & chunk_set)
            for chunk_set in chunk_words
        ]

        best_idx = max(range(n_chunks), key=lambda i: scores[i])
        assigned[best_idx].append(sentence)

    examples = []

    for chunk_idx, source_ids in enumerate(chunks):
        target_segments = make_target_segments(assigned[chunk_idx])

        for target_ids in target_segments:
            examples.append({
                "id": str(row["id"]),
                "chunk_idx": chunk_idx,
                "source_ids": source_ids,
                "target_ids": target_ids
            })

    return examples


doc_ids = sorted(set(str(row["id"]) for row in train_data))
random.Random(SEED).shuffle(doc_ids)

val_count = max(1, int(len(doc_ids) * 0.10))

valid_ids = set(doc_ids[:val_count])
train_ids = set(doc_ids[val_count:])

assert train_ids.isdisjoint(valid_ids)

print("Train documents:", len(train_ids))
print("Validation documents:", len(valid_ids))

# train_examples = []
# valid_examples = []

# for row in tqdm(train_data, desc="Preparing training documents"):
#     examples = build_document_examples(row)

#     if str(row["id"]) in valid_ids:
#         valid_examples.extend(examples)
#     else:
#         train_examples.extend(examples)


test_chunks = []

for row in tqdm(test_data, desc="Preparing test documents"):
    chunks, _ = encode_document_chunks(row["document"])

    for chunk_idx, source_ids in enumerate(chunks):
        test_chunks.append({
            "id": str(row["id"]),
            "chunk_idx": chunk_idx,
            "source_ids": source_ids
        })


print("\nPreprocessing complete")
print("Training examples:", len(train_examples))
print("Validation examples:", len(valid_examples))
print("Test chunks:", len(test_chunks))

sample = train_examples[0]

print("\nSample ID:", sample["id"])
print("Source length:", len(sample["source_ids"]))
print("Target length:", len(sample["target_ids"]))

max_length = max(
    len(PROMPT_IDS) + len(x["source_ids"]) +
    len(SUMMARY_PREFIX_IDS) + len(x["target_ids"])
    for x in train_examples + valid_examples
)

print("Maximum sequence length:", max_length)
print("Train/validation IDs disjoint:", train_ids.isdisjoint(valid_ids))


Tokenizer loaded: ByteLevelBPE
Vocabulary: 32000
Train documents: 7030
Test documents: 100
Train documents: 6327
Validation documents: 703


Preparing test documents:   0%|          | 0/100 [00:00<?, ?it/s]


Preprocessing complete


NameError: name 'train_examples' is not defined

In [11]:
import torch

M2_CHECKPOINT = "/kaggle/working/m2_checkpoints/best.pt"

ckpt = torch.load(M2_CHECKPOINT,map_location="cpu",weights_only=False)

model = M2Model(
    vocab_size=32000,
    d_model=640,
    n_layers=18,
    n_heads=10,
    d_ff=2560,
    dropout=0.05
)

model.load_state_dict(ckpt["model"],strict=True)

print("M2 checkpoint loaded successfully")
print("Step:",ckpt["step"])
print("Validation loss:",ckpt["val_loss"])

del ckpt

M2 checkpoint loaded successfully
Step: 30518
Validation loss: 1.4605953693389893


In [12]:
def apply_lora(model):
    for p in model.parameters():
        p.requires_grad = False

    count = 0

    for block in model.blocks:
        attn = block.attn

        attn.q_proj = LoRALinear(attn.q_proj,LORA_R,LORA_ALPHA,LORA_DROPOUT)
        attn.k_proj = LoRALinear(attn.k_proj,LORA_R,LORA_ALPHA,LORA_DROPOUT)
        attn.v_proj = LoRALinear(attn.v_proj,LORA_R,LORA_ALPHA,LORA_DROPOUT)
        attn.out_proj = LoRALinear(attn.out_proj,LORA_R,LORA_ALPHA,LORA_DROPOUT)

        count += 4

    return count


count = apply_lora(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("LoRA modules:",count)
print("Total parameters:",total_params)
print("Trainable parameters:",trainable_params)
print("Trainable percentage:",round(100*trainable_params/total_params,4),"%")

LoRA modules: 72
Total parameters: 96358912
Trainable parameters: 571392
Trainable percentage: 0.593 %


In [14]:
IGNORE_INDEX = -100

def prepare_example(example):
    source = example["source_ids"]
    target = example["target_ids"]

    input_ids = source + target
    labels = input_ids.copy()

    labels[:len(source)] = [IGNORE_INDEX] * len(source)

    input_ids = input_ids[:-1]
    labels = labels[1:]

    return {
        "input_ids": input_ids,
        "labels": labels
    }


sample = prepare_example(train_examples[0])

print("Input length:",len(sample["input_ids"]))
print("Label length:",len(sample["labels"]))
print("Masked labels:",sample["labels"].count(IGNORE_INDEX))
print("Target tokens used for loss:",sum(x != IGNORE_INDEX for x in sample["labels"]))
print("First non-masked label index:",next(i for i,x in enumerate(sample["labels"]) if x != IGNORE_INDEX))
print("Last label:",sample["labels"][-1])

Input length: 1352
Label length: 1352
Masked labels: 1036
Target tokens used for loss: 316
First non-masked label index: 1036
Last label: 0


In [15]:
from torch.utils.data import Dataset

class SummarizationDataset(Dataset):
    def __init__(self,examples):
        self.data = []
        for example in tqdm(examples,desc="Preparing examples"):
            self.data.append(prepare_example(example))

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        return self.data[idx]


train_dataset = SummarizationDataset(train_examples)
valid_dataset = SummarizationDataset(valid_examples)

print("Training examples:",len(train_dataset))
print("Validation examples:",len(valid_dataset))

Preparing examples:   0%|          | 0/44937 [00:00<?, ?it/s]

Preparing examples:   0%|          | 0/5017 [00:00<?, ?it/s]

Training examples: 44937
Validation examples: 5017


In [16]:
import os
import torch

DATA_DIR = "/kaggle/working/phase6_m2_data"
os.makedirs(DATA_DIR,exist_ok=True)

torch.save(train_dataset.data,os.path.join(DATA_DIR,"train.pt"))
torch.save(valid_dataset.data,os.path.join(DATA_DIR,"valid.pt"))

print("Saved training examples:",len(train_dataset))
print("Saved validation examples:",len(valid_dataset))
print("Files:",os.listdir(DATA_DIR))

Saved training examples: 44937
Saved validation examples: 5017
Files: ['valid.pt', 'test_chunk_predictions.pt', 'train.pt', 'test_chunks.pt']


In [17]:
model_file = "/kaggle/working/m2_model.py"

with open(model_file,"r") as f:
    code = f.read()

old_k = "k = k.expand(B, self.n_heads, T, self.head_dim)"
old_v = "v = v.expand(B, self.n_heads, T, self.head_dim)"

new_k = "k = k.expand(B, self.n_heads, T, self.head_dim).contiguous()"
new_v = "v = v.expand(B, self.n_heads, T, self.head_dim).contiguous()"

assert code.count(old_k) == 1, "K expansion line not found or appears more than once"
assert code.count(old_v) == 1, "V expansion line not found or appears more than once"

code = code.replace(old_k,new_k)
code = code.replace(old_v,new_v)

with open(model_file,"w") as f:
    f.write(code)

print("M2 attention updated successfully.")

with open(model_file,"r") as f:
    lines = f.readlines()

for i in range(53,58):
    print(f"{i+1}: {lines[i].rstrip()}")

M2 attention updated successfully.
54: 
55:         k = k.expand(B, self.n_heads, T, self.head_dim).contiguous().contiguous()
56:         v = v.expand(B, self.n_heads, T, self.head_dim).contiguous().contiguous()
57: 
58:         pos = torch.arange(T, device=x.device)


In [18]:
import torch
from m2_model import M2Model

device = torch.device("cuda:0")

model = M2Model(
    vocab_size=32000,
    d_model=640,
    n_layers=18,
    n_heads=10,
    d_ff=2560,
    dropout=0.05
).to(device)

checkpoint = torch.load(
    "/kaggle/working/m2_checkpoints/best.pt",
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(checkpoint["model"],strict=True)

model.eval()

print("Fresh M2 model loaded successfully.")
print("Parameters:",sum(p.numel() for p in model.parameters()))

Fresh M2 model loaded successfully.
Parameters: 95787520


In [8]:
model.eval()

total_loss = 0.0
total_tokens = 0

with torch.inference_mode():
    for i in tqdm(range(5017),desc="M2 validation test"):
        example = valid_data[i]

        input_ids = torch.tensor(
            [example["input_ids"]],dtype=torch.long,device=device
        )
        labels = torch.tensor(
            [example["labels"]],dtype=torch.long,device=device
        )

        output_positions = torch.nonzero(labels[0] != -100).flatten()
        target_labels = labels.index_select(1,output_positions)

        with torch.autocast("cuda",dtype=torch.float16):
            x = model.token_emb(input_ids)

            for block in model.blocks:
                x = block(x)

            x = model.final_ln(x)
            x = x.index_select(1,output_positions)
            logits = model.lm_head(x)

            loss = F.cross_entropy(
                logits.reshape(-1,logits.size(-1)),
                target_labels.reshape(-1),
                reduction="sum"
            )

        torch.cuda.synchronize()
        total_loss += loss.item()
        total_tokens += target_labels.numel()

print("All example validation PASSED")
print("Average loss:",total_loss/total_tokens)
print("Total tokens:",total_tokens)

model.eval()

M2 validation test:   0%|          | 0/5017 [00:00<?, ?it/s]

All example validation PASSED
Average loss: 1.29343109910743
Total tokens: 1210271


M2Model(
  (token_emb): Embedding(32000, 640)
  (blocks): ModuleList(
    (0-17): 18 x M2Block(
      (attn): MQA(
        (q_proj): Linear(in_features=640, out_features=640, bias=False)
        (k_proj): Linear(in_features=640, out_features=64, bias=False)
        (v_proj): Linear(in_features=640, out_features=64, bias=False)
        (out_proj): Linear(in_features=640, out_features=640, bias=False)
      )
      (ln1): LayerNorm((640,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((640,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=640, out_features=2560, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=2560, out_features=640, bias=True)
      )
      (dropout): Dropout(p=0.05, inplace=False)
    )
  )
  (final_ln): LayerNorm((640,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=640, out_features=32000, bias=False)
)

In [13]:
%%writefile /kaggle/working/train_m2_phase6.py
import os
import shutil
import math
import random
import argparse
import types
import numpy as np
from contextlib import nullcontext
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.distributed as dist
import torch.nn.functional as F

from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset,DataLoader
from torch.utils.data.distributed import DistributedSampler


def setup():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    return local_rank


def cleanup():
    if dist.is_initialized():
        dist.destroy_process_group()


class SummarizationDataset(Dataset):
    def __init__(self,data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        return self.data[idx]


class LoRALinear(nn.Module):
    def __init__(self,base_layer,r=8,alpha=16,dropout=0.05):
        super().__init__()
        self.base = base_layer
        self.base.weight.requires_grad = False

        if self.base.bias is not None:
            self.base.bias.requires_grad = False

        self.scaling = alpha/r
        self.lora_dropout = nn.Dropout(dropout)
        self.lora_A = nn.Parameter(torch.empty(r,base_layer.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base_layer.out_features,r))

        nn.init.kaiming_uniform_(self.lora_A,a=math.sqrt(5))

    def forward(self,x):
        return self.base(x)+(self.lora_dropout(x) @ self.lora_A.T @ self.lora_B.T)*self.scaling


def apply_lora(model):
    for p in model.parameters():
        p.requires_grad = False

    count = 0

    for block in model.blocks:
        attn = block.attn
        attn.q_proj = LoRALinear(attn.q_proj)
        attn.k_proj = LoRALinear(attn.k_proj)
        attn.v_proj = LoRALinear(attn.v_proj)
        attn.out_proj = LoRALinear(attn.out_proj)
        count += 4

    assert count == 72

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert trainable == 571392

    return count


def target_only_forward(self,input_ids,output_positions):
    x = self.token_emb(input_ids)

    for block in self.blocks:
        x = block(x)

    x = self.final_ln(x)
    x = x.index_select(1,output_positions)

    return self.lm_head(x)


def load_model(checkpoint_path):
    from m2_model import M2Model

    model = M2Model(
        vocab_size=32000,
        d_model=640,
        n_layers=18,
        n_heads=10,
        d_ff=2560,
        dropout=0.05
    )

    ckpt = torch.load(checkpoint_path,map_location="cpu",weights_only=False)
    model.load_state_dict(ckpt["model"],strict=True)
    del ckpt

    apply_lora(model)
    model.forward = types.MethodType(target_only_forward,model)

    return model


def collate_fn(batch):
    input_ids = torch.tensor([x["input_ids"] for x in batch],dtype=torch.long)
    labels = torch.tensor([x["labels"] for x in batch],dtype=torch.long)
    return input_ids,labels


def get_cuda_rng_states(device,world_size):
    local_state = torch.cuda.get_rng_state(device).to(device)
    states = [torch.empty_like(local_state) for _ in range(world_size)]
    dist.all_gather(states,local_state)
    return [state.cpu() for state in states]


def save_checkpoint(
    path,
    model,
    optimizer,
    scaler,
    epoch,
    next_batch,
    pending_validation,
    stop_training,
    global_step,
    best_val,
    device,
    rank,
    world_size
):
    cuda_rng_states = get_cuda_rng_states(device,world_size)

    if rank == 0:
        state = {
            "epoch":epoch,
            "next_batch":next_batch,
            "pending_validation":pending_validation,
            "stop_training":stop_training,
            "global_step":global_step,
            "best_val":best_val,
            "model":model.module.state_dict(),
            "optimizer":optimizer.state_dict(),
            "scaler":scaler.state_dict(),
            "cuda_rng_states":cuda_rng_states
        }

        temp_path = path+".tmp"
        torch.save(state,temp_path)
        os.replace(temp_path,path)

    dist.barrier()


def evaluate(model,loader,device,max_batches=0):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    with torch.inference_mode():
        for batch_idx,(input_ids,labels) in enumerate(tqdm(loader,desc="Validation")):
            if max_batches > 0 and batch_idx >= max_batches:
                break

            input_ids = input_ids.to(device,non_blocking=True)
            labels = labels.to(device,non_blocking=True)

            output_positions = torch.nonzero(
                labels[0] != -100,
                as_tuple=False
            ).flatten()

            target_labels = labels.index_select(1,output_positions)

            with torch.autocast("cuda",dtype=torch.float16):
                logits = model(input_ids,output_positions)

                loss = F.cross_entropy(
                    logits.reshape(-1,logits.size(-1)),
                    target_labels.reshape(-1),
                    reduction="sum"
                )

            torch.cuda.synchronize(device)
            total_loss += loss.item()
            total_tokens += target_labels.numel()

    model.train()

    return total_loss/total_tokens


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs",type=int,default=2)
    parser.add_argument("--lr",type=float,default=2e-4)
    parser.add_argument("--accum",type=int,default=9)
    parser.add_argument("--max_steps",type=int,default=0)
    parser.add_argument("--max_val_batches",type=int,default=0)
    parser.add_argument("--save_every_steps",type=int,default=500)
    parser.add_argument("--resume",action="store_true")
    parser.add_argument("--save_dir",type=str,default="/kaggle/working/phase6_m2_checkpoints")
    args = parser.parse_args()

    if args.accum < 1:
        raise ValueError("--accum must be at least 1")

    if args.save_every_steps < 1:
        raise ValueError("--save_every_steps must be at least 1")

    local_rank = setup()
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    device = torch.device("cuda",local_rank)

    seed = 42+rank
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    train_data = torch.load(
        "/kaggle/working/phase6_m2_data/train.pt",
        map_location="cpu",
        weights_only=False
    )

    valid_data = torch.load(
        "/kaggle/working/phase6_m2_data/valid.pt",
        map_location="cpu",
        weights_only=False
    )

    train_dataset = SummarizationDataset(train_data)
    valid_dataset = SummarizationDataset(valid_data)

    train_sampler = DistributedSampler(
        train_dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        seed=42,
        drop_last=True
    )

    loader_generator = torch.Generator()
    loader_generator.manual_seed(12345+rank)

    train_loader = DataLoader(
        train_dataset,
        batch_size=1,
        sampler=train_sampler,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True,
        generator=loader_generator
    )

    valid_loader = None

    if rank == 0:
        valid_loader = DataLoader(
            valid_dataset,
            batch_size=1,
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=0,
            pin_memory=True
        )

        print("Training examples:",len(train_dataset))
        print("Validation examples:",len(valid_dataset))
        print("Steps per epoch:",len(train_loader))
        print("Loading M2 checkpoint...")

    model = load_model(
        "/kaggle/working/m2_checkpoints/best.pt"
    ).to(device)

    model = DDP(
        model,
        device_ids=[local_rank],
        output_device=local_rank,
        find_unused_parameters=False
    )

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=args.lr,
        weight_decay=0.01,
        fused=True
    )

    scaler = torch.amp.GradScaler("cuda")

    steps_per_epoch = math.ceil(len(train_loader)/args.accum)
    total_steps = steps_per_epoch*args.epochs

    global_step = 0
    best_val = float("inf")
    start_epoch = 0
    start_batch = 0
    pending_validation = False
    stop_training = False

    save_dir = args.save_dir
    latest_path = os.path.join(save_dir,"latest.pt")

    if rank == 0:
        os.makedirs(save_dir,exist_ok=True)

    dist.barrier()

    if args.resume:
        if not os.path.exists(latest_path):
            raise FileNotFoundError(
                "Cannot resume: latest checkpoint not found at "+latest_path
            )

        resume_state = torch.load(
            latest_path,
            map_location="cpu",
            weights_only=False
        )

        model.module.load_state_dict(resume_state["model"],strict=True)
        optimizer.load_state_dict(resume_state["optimizer"])
        scaler.load_state_dict(resume_state["scaler"])

        start_epoch = resume_state["epoch"]
        start_batch = resume_state.get("next_batch",0)
        pending_validation = resume_state.get("pending_validation",False)
        stop_training = resume_state.get("stop_training",False)
        global_step = resume_state["global_step"]
        best_val = resume_state["best_val"]

        if "cuda_rng_states" in resume_state:
            torch.cuda.set_rng_state(
                resume_state["cuda_rng_states"][rank],
                device=device
            )

        if rank == 0:
            print("Resuming from checkpoint.")
            print("Epoch:",start_epoch+1)
            print("Next batch:",start_batch)
            print("Global optimizer step:",global_step)
            print("Pending validation:",pending_validation)

        del resume_state

    else:
        if os.path.exists(latest_path):
            raise FileExistsError(
                "A Phase 6 checkpoint already exists. Use --resume to continue "
                "or choose a new --save_dir to start a separate run."
            )

    if stop_training and not pending_validation:
        if rank == 0:
            print("The saved run had already reached its stopping condition.")
        cleanup()
        return

    for epoch in range(start_epoch,args.epochs):
        model.train()
        train_sampler.set_epoch(epoch)

        if epoch == start_epoch and pending_validation:
            epoch_finished = True
        else:
            epoch_finished = False

        if not epoch_finished:
            optimizer.zero_grad(set_to_none=True)
            running_loss = 0.0

            progress = tqdm(
                enumerate(train_loader),
                total=len(train_loader),
                desc=f"Epoch {epoch+1}/{args.epochs}",
                disable=rank!=0
            )

            for step,(input_ids,labels) in progress:
                if epoch == start_epoch and step < start_batch:
                    continue

                group_start = (step//args.accum)*args.accum
                group_size = min(args.accum,len(train_loader)-group_start)

                sync_now = (
                    (step+1)%args.accum == 0
                    or (step+1)==len(train_loader)
                )

                sync_context = nullcontext() if sync_now else model.no_sync()

                input_ids = input_ids.to(device,non_blocking=True)
                labels = labels.to(device,non_blocking=True)

                output_positions = torch.nonzero(
                    labels[0] != -100,
                    as_tuple=False
                ).flatten()

                target_labels = labels.index_select(1,output_positions)

                with sync_context:
                    with torch.autocast("cuda",dtype=torch.float16):
                        logits = model(input_ids,output_positions)

                        raw_loss = F.cross_entropy(
                            logits.reshape(-1,logits.size(-1)),
                            target_labels.reshape(-1)
                        )

                        loss = raw_loss/group_size

                    scaler.scale(loss).backward()

                running_loss += raw_loss.detach().item()

                if sync_now:
                    scaler.unscale_(optimizer)

                    torch.nn.utils.clip_grad_norm_(
                        [p for p in model.parameters() if p.requires_grad],
                        1.0
                    )

                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

                    global_step += 1

                    lr = args.lr*(1-global_step/total_steps)
                    lr = max(lr,args.lr*0.05)

                    for group in optimizer.param_groups:
                        group["lr"] = lr

                    if global_step % args.save_every_steps == 0:
                        save_checkpoint(
                            latest_path,
                            model,
                            optimizer,
                            scaler,
                            epoch,
                            step+1,
                            False,
                            stop_training,
                            global_step,
                            best_val,
                            device,
                            rank,
                            world_size
                        )

                        if rank == 0:
                            print(
                                f"\nCheckpoint saved at optimizer step {global_step}, "
                                f"epoch {epoch+1}, next batch {step+1}."
                            )

                if rank == 0:
                    progress.set_postfix(
                        loss=running_loss/(step-start_batch+1),
                        lr=optimizer.param_groups[0]["lr"]
                    )

                if args.max_steps > 0 and global_step >= args.max_steps:
                    stop_training = True
                    break

            # Save the completed epoch before starting validation.
            save_checkpoint(
                latest_path,
                model,
                optimizer,
                scaler,
                epoch,
                len(train_loader),
                True,
                stop_training,
                global_step,
                best_val,
                device,
                rank,
                world_size
            )

            if rank == 0:
                print(f"\nEpoch {epoch+1} training checkpoint saved before validation.")

        # All ranks reach this point. Rank 0 performs validation.
        dist.barrier()

        if rank == 0:
            val_loss = evaluate(
                model.module,
                valid_loader,
                device,
                args.max_val_batches
            )

            print(f"\nEpoch {epoch+1}: validation loss = {val_loss:.5f}")

            improved = val_loss < best_val

            if improved:
                best_val = val_loss

        # Save the post-validation state and advance to the next epoch.
        save_checkpoint(
            latest_path,
            model,
            optimizer,
            scaler,
            epoch+1,
            0,
            False,
            stop_training,
            global_step,
            best_val,
            device,
            rank,
            world_size
        )

        if rank == 0:
            if improved:
                best_path = os.path.join(save_dir,"best.pt")
                temp_path = best_path+".tmp"
                shutil.copyfile(latest_path,temp_path)
                os.replace(temp_path,best_path)
                print("Best checkpoint updated.")

            print("Latest checkpoint saved.")

        dist.barrier()

        pending_validation = False
        start_batch = 0

        if stop_training:
            break

    if rank == 0:
        print("\nTraining completed.")

    cleanup()


if __name__ == "__main__":
    main()


Overwriting /kaggle/working/train_m2_phase6.py


In [25]:
!OMP_NUM_THREADS=1 torchrun --standalone --nproc_per_node=2 /kaggle/working/train_m2_phase6.py --epochs 2 --max_steps 2 --max_val_batches 2 --save_dir /kaggle/working/phase6_m2_smoke

Training examples: 44937
Validation examples: 5017
Steps per epoch: 22468
Loading M2 checkpoint...
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
Epoch 1/2:   0%|     | 17/22468 [00:04<1:42:33,  3.65it/s, loss=1.66, lr=0.0002]

Epoch 1 training checkpoint saved before validation.
Validation:   0%|                              | 2/5017 [00:00<11:25,  7.31it/s]

Epoch 1: validation loss = 1.17328
Best checkpoint updated.
Latest checkpoint saved.

Training completed.


In [ ]:
!OMP_NUM_THREADS=1 torchrun --standalone --nproc_per_node=2 /kaggle/working/train_m2_phase6.py --epochs 2 --lr 2e-4 --accum 9

Training examples: 44937
Validation examples: 5017
Steps per epoch: 22468
Loading M2 checkpoint...
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
Epoch 1/2:  20%|▍ | 4499/22468 [17:21<1:07:48,  4.42it/s, loss=1.38, lr=0.00018]
Checkpoint saved at optimizer step 500, epoch 1, next batch 4500.
Epoch 1/2:  40%|█▌  | 8999/22468 [34:46<49:14,  4.56it/s, loss=1.32, lr=0.00016]
Checkpoint saved at optimizer step 1000, epoch 1, next batch 9000.
Epoch 1/2:  60%|██▍ | 13499/22468 [52:14<30:37,  4.88it/s, loss=1.3, lr=0.00014]
Checkpoint saved at optimizer step 1500, epoch 1, next batch 13500.
Epoch 1/2:  80%|▊| 17999/22468 [1:09:38<17:40,  4.21it/s, loss=1.28, lr=0.00012]
Checkpoint saved at optimizer step 2000, epoch 1, next batch 18000.
Epoch 1/2: 100%|██| 22468/22468 [1:26:56<00:00,  4.31it/s, los

In [1]:
!OMP_NUM_THREADS=1 torchrun --standalone --nproc_per_node=2 /kaggle/working/train_m2_phase6.py --epochs 2 --lr 2e-4 --accum 9 --resume

Training examples: 44937
Validation examples: 5017
Steps per epoch: 22468
Loading M2 checkpoint...
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
Resuming from checkpoint.
Epoch: 3
Next batch: 0
Global optimizer step: 4994
Pending validation: False

Training completed.


In [19]:
import torch

train_data=torch.load(
    "/kaggle/working/phase6_m2_data/train.pt",
    map_location="cpu",
    weights_only=False
)

valid_data=torch.load(
    "/kaggle/working/phase6_m2_data/valid.pt",
    map_location="cpu",
    weights_only=False
)

print("Train examples:",len(train_data))
print("Validation examples:",len(valid_data))

print("\nFirst training example:")
sample=train_data[0]

print("Keys:",sample.keys())

input_ids=sample["input_ids"]
labels=sample["labels"]

print("Input length:",len(input_ids))
print("Label length:",len(labels))
print("Non-masked label tokens:",sum(x!=-100 for x in labels))

print("\nInput ID prefix:",input_ids[:40])
print("\nLabel prefix:",labels[:40])

lengths=[len(x["input_ids"]) for x in train_data]
print("\nInput length statistics:")
print("Min:",min(lengths))
print("Max:",max(lengths))
print("Average:",sum(lengths)/len(lengths))

Train examples: 44937
Validation examples: 5017

First training example:
Keys: dict_keys(['input_ids', 'labels'])
Input length: 1352
Label length: 1352
Non-masked label tokens: 316

Input ID prefix: [2239, 33, 796, 33, 3302, 2608, 33, 4148, 17211, 33, 266, 33, 10179, 33, 2203, 3195, 33, 258, 33, 899, 33, 1060, 33, 447, 33, 27061, 10228, 33, 266, 33, 258, 33, 5306, 33, 1660, 33, 887, 33, 399, 33]

Label prefix: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

Input length statistics:
Min: 52
Max: 1899
Average: 1542.8097781338317


In [21]:
import os
import json
import torch
from tqdm.auto import tqdm

TEST_CHUNK_PATH = "/kaggle/working/phase6_m2_data/test_chunks.pt"

test_chunks = []

for row in tqdm(test_data,desc="Preparing M2 test chunks"):
    chunks,_ = encode_document_chunks(row["document"])

    for chunk_idx,source_ids in enumerate(chunks):
        test_chunks.append({
            "id": str(row["id"]),
            "chunk_idx": chunk_idx,
            "source_ids": source_ids
        })

torch.save(test_chunks,TEST_CHUNK_PATH)

print("Test documents:",len(test_data))
print("Test chunks:",len(test_chunks))
print("Saved:",TEST_CHUNK_PATH)

print("\nFirst chunk:")
print("ID:",test_chunks[0]["id"])
print("Chunk index:",test_chunks[0]["chunk_idx"])
print("Source length:",len(test_chunks[0]["source_ids"]))
print("Maximum source length:",max(len(x["source_ids"]) for x in test_chunks))

Preparing M2 test chunks:   0%|          | 0/100 [00:00<?, ?it/s]

Test documents: 100
Test chunks: 816
Saved: /kaggle/working/phase6_m2_data/test_chunks.pt

First chunk:
ID: 4963
Chunk index: 0
Source length: 1156
Maximum source length: 1400


In [13]:
import sys
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0,"/kaggle/working")

from m2_model import M2Model

# --------------------------------------------------
# 1. LoRA layer (same as training)
# --------------------------------------------------

class LoRALinear(nn.Module):
    def __init__(self,base_layer,r=8,alpha=16,dropout=0.05):
        super().__init__()
        self.base = base_layer
        self.base.weight.requires_grad = False

        if self.base.bias is not None:
            self.base.bias.requires_grad = False

        self.scaling = alpha/r
        self.lora_dropout = nn.Dropout(dropout)
        self.lora_A = nn.Parameter(torch.empty(r,base_layer.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base_layer.out_features,r))

        nn.init.kaiming_uniform_(self.lora_A,a=math.sqrt(5))

    def forward(self,x):
        return self.base(x)+(self.lora_dropout(x) @ self.lora_A.T @ self.lora_B.T)*self.scaling


def apply_lora(model):
    for p in model.parameters():
        p.requires_grad = False

    for block in model.blocks:
        attn = block.attn
        attn.q_proj = LoRALinear(attn.q_proj)
        attn.k_proj = LoRALinear(attn.k_proj)
        attn.v_proj = LoRALinear(attn.v_proj)
        attn.out_proj = LoRALinear(attn.out_proj)


# --------------------------------------------------
# 2. Load Phase 6 best checkpoint
# --------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = M2Model(
    vocab_size=32000,
    d_model=640,
    n_layers=18,
    n_heads=10,
    d_ff=2560,
    dropout=0.05
)

apply_lora(model)

checkpoint_path = "/kaggle/working/phase6_m2_checkpoints/best.pt"
ckpt = torch.load(checkpoint_path,map_location="cpu",weights_only=False)

model.load_state_dict(ckpt["model"],strict=True)
del ckpt

model = model.to(device)
model.eval()

print("Model loaded successfully on",device)


# --------------------------------------------------
# 3. Cached attention
# --------------------------------------------------

def cached_attention_prefill(attn,x):
    B,T,C = x.shape

    q = attn.q_proj(x).view(
        B,T,attn.n_heads,attn.head_dim
    ).transpose(1,2)

    k = attn.k_proj(x).view(
        B,T,1,attn.head_dim
    ).transpose(1,2)

    v = attn.v_proj(x).view(
        B,T,1,attn.head_dim
    ).transpose(1,2)

    pos = torch.arange(T,device=x.device)
    relative = pos[None,:]-pos[:,None]

    slopes = attn.alibi_slopes.to(
        device=x.device,dtype=torch.float32
    )

    bias = slopes[None,:,None,None]*relative[
        None,None,:,: 
    ].float()

    bias = bias.masked_fill(
        relative[None,None,:,:]>0,float("-inf")
    )

    k_expanded = k.expand(B,attn.n_heads,T,attn.head_dim)
    v_expanded = v.expand(B,attn.n_heads,T,attn.head_dim)

    out = F.scaled_dot_product_attention(
        q,k_expanded,v_expanded,
        attn_mask=bias.to(dtype=q.dtype),
        dropout_p=0.0
    )

    out = out.transpose(1,2).contiguous().view(B,T,C)

    return attn.out_proj(out),(k,v)


def cached_attention_step(attn,x,cache):
    B,T,C = x.shape

    q = attn.q_proj(x).view(
        B,T,attn.n_heads,attn.head_dim
    ).transpose(1,2)

    k = attn.k_proj(x).view(
        B,T,1,attn.head_dim
    ).transpose(1,2)

    v = attn.v_proj(x).view(
        B,T,1,attn.head_dim
    ).transpose(1,2)

    if cache is not None:
        old_k,old_v = cache
        k = torch.cat((old_k,k),dim=2)
        v = torch.cat((old_v,v),dim=2)

    total_len = k.size(2)
    query_pos = total_len-1

    key_pos = torch.arange(total_len,device=x.device)
    relative = key_pos-query_pos

    slopes = attn.alibi_slopes.to(
        device=x.device,dtype=torch.float32
    )

    bias = slopes[None,:,None,None]*relative[
        None,None,None,: 
    ].float()

    k_expanded = k.expand(B,attn.n_heads,total_len,attn.head_dim)
    v_expanded = v.expand(B,attn.n_heads,total_len,attn.head_dim)

    out = F.scaled_dot_product_attention(
        q,k_expanded,v_expanded,
        attn_mask=bias.to(dtype=q.dtype),
        dropout_p=0.0
    )

    out = out.transpose(1,2).contiguous().view(B,T,C)

    return attn.out_proj(out),(k,v)


# --------------------------------------------------
# 4. Cached transformer block
# --------------------------------------------------

def cached_block_prefill(block,x):
    attn_out,cache = cached_attention_prefill(block.attn,x)

    x = block.ln1(x+block.dropout(attn_out))
    x = block.ln2(x+block.dropout(block.ffn(x)))

    return x,cache


def cached_block_step(block,x,cache):
    attn_out,new_cache = cached_attention_step(
        block.attn,x,cache
    )

    x = block.ln1(x+block.dropout(attn_out))
    x = block.ln2(x+block.dropout(block.ffn(x)))

    return x,new_cache


# --------------------------------------------------
# 5. Cached model prefill and one-token decoding
# --------------------------------------------------

def cached_prefill(model,input_ids):
    x = model.token_emb(input_ids)
    caches = []

    for block in model.blocks:
        x,cache = cached_block_prefill(block,x)
        caches.append(cache)

    logits = model.lm_head(model.final_ln(x[:,-1:,:]))

    return logits,caches


def cached_decode_step(model,input_token,caches):
    x = model.token_emb(input_token)
    new_caches = []

    for i,block in enumerate(model.blocks):
        x,cache = cached_block_step(block,x,caches[i])
        new_caches.append(cache)

    logits = model.lm_head(model.final_ln(x))

    return logits,new_caches


# --------------------------------------------------
# 6. Verify cached vs original model
# --------------------------------------------------

# Use the first prepared test chunk
test_source = test_chunks[0]["source_ids"]

source = torch.tensor(
    [test_source],dtype=torch.long,device=device
)

# Append one token to test incremental decoding
next_token = torch.tensor(
    [[0]],dtype=torch.long,device=device
)

with torch.inference_mode():

    # Original model: full sequence
    full_input = torch.cat((source,next_token),dim=1)

    with torch.autocast("cuda",dtype=torch.float16):
        full_logits = model(full_input)
        original_next_logits = full_logits[:,-1,:]

    # Cached model: prefill source, then decode one token
    with torch.autocast("cuda",dtype=torch.float16):
        prefill_logits,caches = cached_prefill(model,source)
        cached_next_logits,new_caches = cached_decode_step(
            model,next_token,caches
        )

    original_next_logits = original_next_logits.float()
    cached_next_logits = cached_next_logits[:,-1,:].float()

    max_diff = (
        original_next_logits-cached_next_logits
    ).abs().max().item()

    original_token = original_next_logits.argmax(dim=-1).item()
    cached_token = cached_next_logits.argmax(dim=-1).item()

print("Original next token:",original_token)
print("Cached next token:",cached_token)
print("Maximum logit difference:",max_diff)
print("Greedy token match:",original_token == cached_token)

Model loaded successfully on cuda
Original next token: 8401
Cached next token: 8401
Maximum logit difference: 0.0078125
Greedy token match: True


In [21]:
import os
import torch
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Generation settings
# --------------------------------------------------

EOS_ID = 0
MAX_NEW_TOKENS = 500

PRED_PATH = "/kaggle/working/phase6_m2_data/test_chunk_predictions.pt"

# Resume from existing predictions if available
if os.path.exists(PRED_PATH):
    predictions = torch.load(PRED_PATH,map_location="cpu",weights_only=False)
    completed = set((x["id"],x["chunk_idx"]) for x in predictions)
    print("Resuming:",len(completed),"chunks already completed")
else:
    predictions = []
    completed = set()

# --------------------------------------------------
# 2. Cached greedy generation
# --------------------------------------------------

@torch.inference_mode()
def generate_chunk(source_ids):
    source = torch.tensor([source_ids],dtype=torch.long,device=device)
    generated = []
    with torch.autocast("cuda",dtype=torch.float16):
        logits,caches = cached_prefill(model,source)
        next_token = logits[:,-1,:].argmax(dim=-1).item()
        if next_token == EOS_ID:
            return generated

        generated.append(next_token)
        for step in range(1,MAX_NEW_TOKENS):
            input_token = torch.tensor([[next_token]],dtype=torch.long,device=device)
            logits,caches = cached_decode_step(model,input_token,caches)
            next_token = logits[:,-1,:].argmax(dim=-1).item()
            if next_token == EOS_ID:
                break

            generated.append(next_token)

    return generated


# --------------------------------------------------
# 3. Generate all test chunks
# --------------------------------------------------

model.eval()

for i,chunk in enumerate(tqdm(test_chunks,desc="Generating summaries")):

    key = (str(chunk["id"]),chunk["chunk_idx"])

    if key in completed:
        continue

    generated_ids = generate_chunk(chunk["source_ids"])

    predictions.append({
        "id":str(chunk["id"]),
        "chunk_idx":chunk["chunk_idx"],
        "summary_ids":generated_ids
    })

    completed.add(key)

    # Save every 10 chunks
    if len(predictions)%10 == 0:
        torch.save(predictions,PRED_PATH)

# Final save
torch.save(predictions,PRED_PATH)

print("\nGeneration completed.")
print("Predicted chunks:",len(predictions))
print("Saved at:",PRED_PATH)

# Quick sample
print("\nFirst prediction:")
print("ID:",predictions[0]["id"])
print("Chunk:",predictions[0]["chunk_idx"])
print("Generated tokens:",len(predictions[0]["summary_ids"]))

Resuming: 520 chunks already completed


Generating summaries:   0%|          | 0/816 [00:00<?, ?it/s]

AcceleratorError: CUDA error: an illegal memory access was encountered
Search for `cudaErrorIllegalAddress' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [19]:
EOS_ID = 0
MAX_NEW_TOKENS = 500
@torch.inference_mode()
def generate_chunk(source_ids):
    source = torch.tensor([source_ids],dtype=torch.long,device=device)
    generated = []
    with torch.autocast("cuda",dtype=torch.float16):
        logits,caches = cached_prefill(model,source)
        next_token = logits[:,-1,:].argmax(dim=-1).item()
        if next_token == EOS_ID:
            return generated

        generated.append(next_token)
        for step in range(1,MAX_NEW_TOKENS):
            input_token = torch.tensor([[next_token]],dtype=torch.long,device=device)
            logits,caches = cached_decode_step(model,input_token,caches)
            next_token = logits[:,-1,:].argmax(dim=-1).item()
            if next_token == EOS_ID:
                break

            generated.append(next_token)

    return generated

In [20]:
import torch
from tqdm.auto import tqdm

# Use stable attention backend
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

PRED_PATH = "/kaggle/working/phase6_m2_data/test_chunk_predictions.pt"

# Load saved predictions
predictions = torch.load(
    PRED_PATH,
    map_location="cpu",
    weights_only=False
)

completed = set(
    (str(x["id"]),x["chunk_idx"]) for x in predictions
)

print("Resuming from:",len(completed))
model.eval()

new_count = 0

for chunk in tqdm(test_chunks,desc="Generating remaining summaries"):

    key = (str(chunk["id"]),chunk["chunk_idx"])

    if key in completed:
        continue

    generated_ids = generate_chunk(chunk["source_ids"])

    predictions.append({
        "id":str(chunk["id"]),
        "chunk_idx":chunk["chunk_idx"],
        "summary_ids":generated_ids
    })

    completed.add(key)
    new_count += 1

    if new_count % 5 == 0:
        torch.save(predictions,PRED_PATH)

# Final save
torch.save(predictions,PRED_PATH)

print("\nGeneration completed.")
print("New predictions:",new_count)
print("Total predictions:",len(predictions))
print("Remaining:",len(test_chunks)-len(completed))

Resuming from: 525


Generating remaining summaries:   0%|          | 0/816 [00:00<?, ?it/s]


Generation completed.
New predictions: 291
Total predictions: 816
Remaining: 0


In [23]:
import os
import json
import zipfile
import torch
from collections import defaultdict
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Load predictions
# --------------------------------------------------

PRED_PATH = "/kaggle/working/phase6_m2_data/test_chunk_predictions.pt"

predictions = torch.load(
    PRED_PATH,
    map_location="cpu",
    weights_only=False
)

assert len(predictions) == 816, f"Expected 816 chunks, got {len(predictions)}"

# --------------------------------------------------
# 2. Decode token IDs using ByteLevelBPE
# --------------------------------------------------

def decode_ids(token_ids):
    decoded_bytes = bytearray()

    for token_id in token_ids:
        if token_id == bpe.eos_id:
            break

        if token_id < 1 or token_id > len(bpe._token_bytes):
            continue

        decoded_bytes.extend(bpe._token_bytes[token_id - 1])

    return decoded_bytes.decode("utf-8",errors="replace")


# --------------------------------------------------
# 3. Group chunk summaries by document ID
# --------------------------------------------------

document_chunks = defaultdict(list)

for item in predictions:
    summary_text = decode_ids(item["summary_ids"])

    document_chunks[str(item["id"])].append(
        (item["chunk_idx"],summary_text.strip())
    )

# --------------------------------------------------
# 4. Combine summaries in chunk order
# --------------------------------------------------

final_predictions = []

for doc_id in tqdm(
    [str(row["id"]) for row in test_data],
    desc="Combining document summaries"
):
    chunks = sorted(
        document_chunks[doc_id],
        key=lambda x: x[0]
    )

    summary_parts = [
        text for chunk_idx,text in chunks if text
    ]

    final_summary = "\n\n".join(summary_parts).strip()

    final_predictions.append({
        "id":doc_id,
        "summary":final_summary
    })

# --------------------------------------------------
# 5. Validate submission
# --------------------------------------------------

expected_ids = [str(row["id"]) for row in test_data]
actual_ids = [x["id"] for x in final_predictions]

assert len(final_predictions) == 100
assert actual_ids == expected_ids
assert len(set(actual_ids)) == 100
assert all(isinstance(x["summary"],str) for x in final_predictions)

print("Documents:",len(final_predictions))
print("Empty summaries:",sum(not x["summary"] for x in final_predictions))
print("Total summary characters:",sum(len(x["summary"]) for x in final_predictions))

# --------------------------------------------------
# 6. Save JSON
# --------------------------------------------------

OUTPUT_DIR = "/kaggle/working/submission_m2_p6"
os.makedirs(OUTPUT_DIR,exist_ok=True)

JSON_PATH = os.path.join(OUTPUT_DIR,"prediction.json")

with open(JSON_PATH,"w",encoding="utf-8") as f:
    json.dump(
        final_predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

# --------------------------------------------------
# 7. Create ZIP
# --------------------------------------------------

ZIP_PATH = "/kaggle/working/submission_m2_p6.zip"

with zipfile.ZipFile(
    ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED
) as z:
    z.write(JSON_PATH,arcname="prediction.json")

print("\nSubmission created successfully!")
print("JSON:",JSON_PATH)
print("ZIP:",ZIP_PATH)
print("ZIP size:",round(os.path.getsize(ZIP_PATH)/(1024*1024),2),"MB")

# Preview
print("\nFirst document prediction:")
print("ID:",final_predictions[0]["id"])
print("Summary:",final_predictions[0]["summary"][:1000])

Combining document summaries:   0%|          | 0/100 [00:00<?, ?it/s]

Documents: 100
Empty summaries: 5
Total summary characters: 251427

Submission created successfully!
JSON: /kaggle/working/submission_m2_p6/prediction.json
ZIP: /kaggle/working/submission_m2_p6.zip
ZIP size: 0.03 MB

First document prediction:
ID: 4963
Summary: The assessee appealed to the Appellate Assistant Commissioner and urged that the Appellate Assistant Commissioner had no jurisdiction to include in the net wealth tax assessments of the assessee.The Income-tax Officer sought support from a judgment of the Andhra Pradesh High Court in Vedreva Venkappa Rao vs Commissioner of Wealth Tax ; , referred to.

[311 E F] 2.[311 E F] 3.[311 E F] 3.[311 E F] 3.[311 E F] 4.[311 E F] 4.[311 E F] 5.[311 E F] 5.[311 E F] 6.[311 E F] 6.[311 E F] 6.[311 E F] 6.[311 E F] 6.[311 E F] 6.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.[311 E F] 7.

In [27]:
import copy
from tqdm.auto import tqdm
predictions_before_regen = copy.deepcopy(predictions)
torch.save(
    predictions_before_regen,
    "/kaggle/working/phase6_m2_data/test_chunk_predictions_before_regen.pt"
)
empty_ids = {"1778", "6852", "3019", "4451", "5888"}
bad_keys = set()
for item in predictions:
    key = (str(item["id"]),int(item["chunk_idx"]))
    if len(item["summary_ids"]) == 500:
        bad_keys.add(key)
    if str(item["id"]) in empty_ids:
        bad_keys.add(key)

print("Chunks selected for regeneration:",len(bad_keys))
def generate_with_constraints(
    source_ids,
    max_new_tokens=500,
    min_new_tokens=24,
    repetition_penalty=1.15,
    no_repeat_ngram_size=3
):
    source = torch.tensor([source_ids],dtype=torch.long,device=device)
    generated = []
    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")):
            logits,caches = cached_prefill(model,source)
            for step in range(max_new_tokens):
                scores = logits[0,-1,:].float().clone()
                if len(generated) > 0:
                    previous_tokens = list(set(generated))
                    for token in previous_tokens:
                        if scores[token] < 0:
                            scores[token] *= repetition_penalty
                        else:
                            scores[token] /= repetition_penalty

                # Do not allow EOS before minimum length
                if len(generated) < min_new_tokens:
                    scores[0] = float("-inf")
                if len(generated) >= no_repeat_ngram_size - 1:
                    n = no_repeat_ngram_size
                    prefix = generated[-(n-1):]
                    banned_tokens = []
                    for i in range(len(generated)-n+1):
                        if generated[i:i+n-1] == prefix:
                            banned_tokens.append(generated[i+n-1])
                    if len(banned_tokens) > 0:
                        constrained_scores = scores.clone()
                        constrained_scores[banned_tokens] = float("-inf")
                        # Avoid getting stuck if every candidate is banned
                        if torch.isfinite(constrained_scores).any():
                            scores = constrained_scores
                next_token = scores.argmax().item()
                if next_token == 0:
                    break

                generated.append(next_token)
                input_token = torch.tensor([[next_token]],dtype=torch.long,device=device)
                logits,caches = cached_decode_step(model,input_token,caches)
    return generated
    
chunk_lookup = {(str(item["id"]),int(item["chunk_idx"])): item for item in predictions}
regenerated = 0
for chunk in tqdm(test_chunks,desc="Regenerating chunks"):
    key = (str(chunk["id"]),int(chunk["chunk_idx"]))
    if key not in bad_keys:
        continue

    new_ids = generate_with_constraints(
        chunk["source_ids"],
        max_new_tokens=500,
        min_new_tokens=24,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3
    )

    chunk_lookup[key]["summary_ids"] = new_ids
    regenerated += 1

print("\nRegeneration completed.")
print("Chunks regenerated:",regenerated)
print("Chunks still empty:",sum(len(x["summary_ids"]) == 0 for x in predictions))
print("Chunks still at 500 tokens:",sum(len(x["summary_ids"]) == 500 for x in predictions))
torch.save(predictions,"/kaggle/working/phase6_m2_data/test_chunk_predictions.pt")

Chunks selected for regeneration: 179


Regenerating chunks:   0%|          | 0/816 [00:00<?, ?it/s]


Regeneration completed.
Chunks regenerated: 179
Chunks still empty: 444
Chunks still at 500 tokens: 0


In [31]:
source_lookup = {
    (str(item["id"]),int(item["chunk_idx"])): item["source_ids"]
    for item in test_chunks
}
empty_chunks = [
    item for item in predictions
    if len(item["summary_ids"]) == 0
]

print("Empty chunks to regenerate:",len(empty_chunks))
regenerated = 0
missing_sources = []

for item in tqdm(empty_chunks,desc="Regenerating empty chunks"):
    key = (str(item["id"]),int(item["chunk_idx"]))
    source_ids = source_lookup.get(key)
    if source_ids is None:
        missing_sources.append(key)
        continue

    new_ids = generate_with_constraints(
        source_ids,
        max_new_tokens=500,
        min_new_tokens=24,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3
    )

    item["summary_ids"] = new_ids
    regenerated += 1

torch.save(
    predictions,
    "/kaggle/working/phase6_m2_data/test_chunk_predictions.pt"
)

print("\nRegeneration completed.")
print("Successfully regenerated:",regenerated)
print("Missing source chunks:",len(missing_sources))
print("Remaining empty chunks:",sum(len(x["summary_ids"]) == 0 for x in predictions))

Empty chunks to regenerate: 439


Regenerating empty chunks:   0%|          | 0/439 [00:00<?, ?it/s]


Regeneration completed.
Successfully regenerated: 439
Missing source chunks: 0
Remaining empty chunks: 0


In [32]:
import os
import json
import zipfile
from collections import defaultdict

# --------------------------------------------------
# 1. Combine chunk predictions by document
# --------------------------------------------------

document_chunks = defaultdict(list)

for item in predictions:
    summary_text = decode_ids(item["summary_ids"])

    document_chunks[str(item["id"])].append(
        (item["chunk_idx"],summary_text.strip())
    )

final_predictions = []

for row in test_data:
    doc_id = str(row["id"])

    chunks = sorted(
        document_chunks[doc_id],
        key=lambda x: x[0]
    )

    summary_parts = [
        text for chunk_idx,text in chunks if text
    ]

    final_summary = "\n\n".join(summary_parts).strip()

    final_predictions.append({
        "id": doc_id,
        "summary": final_summary
    })


# --------------------------------------------------
# 2. Validate submission
# --------------------------------------------------

assert len(final_predictions) == len(test_data)
assert len(set(x["id"] for x in final_predictions)) == len(test_data)
assert all(isinstance(x["summary"],str) for x in final_predictions)

empty_docs = [
    x["id"] for x in final_predictions
    if not x["summary"].strip()
]

print("Documents:",len(final_predictions))
print("Empty summaries:",len(empty_docs))
print("Total summary characters:",
      sum(len(x["summary"]) for x in final_predictions))


# --------------------------------------------------
# 3. Save prediction.json
# --------------------------------------------------

output_dir = "/kaggle/working/submission_m2_pr"
os.makedirs(output_dir,exist_ok=True)

json_path = os.path.join(output_dir,"prediction.json")

with open(json_path,"w",encoding="utf-8") as f:
    json.dump(
        final_predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\nJSON saved:",json_path)


# --------------------------------------------------
# 4. Create ZIP
# --------------------------------------------------

zip_path = "/kaggle/working/submission_m2_pr.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:
    zf.write(json_path,arcname="prediction.json")

print("ZIP created:",zip_path)
print("ZIP size:",round(os.path.getsize(zip_path)/(1024*1024),2),"MB")


# --------------------------------------------------
# 5. Verify ZIP contents
# --------------------------------------------------

with zipfile.ZipFile(zip_path,"r") as zf:
    print("\nZIP contents:",zf.namelist())

    with zf.open("prediction.json") as f:
        check = json.load(f)

    assert len(check) == 100
    assert all("id" in x and "summary" in x for x in check)

print("Submission ZIP verified successfully!")

Documents: 100
Empty summaries: 0
Total summary characters: 88270

JSON saved: /kaggle/working/submission_m2_pr/prediction.json
ZIP created: /kaggle/working/submission_m2_pr.zip
ZIP size: 0.03 MB

ZIP contents: ['prediction.json']
Submission ZIP verified successfully!
